In [ ]:
# =============================================================================
# kSZ²-21cm : Lightcone Simulation and Plotting
# =============================================================================

# =============================================================================
# CELL 1: Imports and Setup
# =============================================================================

import numpy as np
import matplotlib as mpl
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

import py21cmfast as p21c
from py21cmfast import plotting

import os
import glob
import time
from datetime import datetime

# PBS vs desktop backend
if os.environ.get('PBS_JOBID'):
    matplotlib.use('Agg')
    print("✓ Using Agg backend (PBS/server mode)")
else:
    matplotlib.use('Agg')   # change to TkAgg for interactive desktop display
    print("✓ Using Agg backend")

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Output and Cache Directories
# =============================================================================

plot_dir = "11May2026_kSZ2_21cm/plots"

if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")

print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")

# --- Cache directory (PBS-aware) ---
try:
    # Running as .py script via PBS
    main_cache_dir = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "cache"
    )
except NameError:
    # Running in Jupyter notebook
    main_cache_dir = os.path.join(
        os.getcwd(), "11May2026_kSZ2_21cm", "cache"
    )

os.makedirs(main_cache_dir, exist_ok=True)
print(f"Cache directory: {main_cache_dir}")

# =============================================================================
# CELL 1b: Global Plot Settings
# DO NOT override font sizes, grid, or tick settings in any downstream cell.
# All plots rely entirely on these settings + PDF_STYLE / PNG_STYLE contexts.
# =============================================================================

plt.rcParams.update({
    # Font
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 20,
    'axes.labelsize'     : 28,
    'axes.titlesize'     : 22,
    'xtick.labelsize'    : 22,
    'ytick.labelsize'    : 22,
    'legend.fontsize'    : 18,
    'figure.titlesize'   : 20,
    # Ticks
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.major.size'   : 6,
    'ytick.major.size'   : 6,
    'xtick.minor.size'   : 3,
    'ytick.minor.size'   : 3,
    'xtick.major.width'  : 1.0,
    'ytick.major.width'  : 1.0,
    'xtick.minor.width'  : 0.8,
    'ytick.minor.width'  : 0.8,
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    # Lines / axes
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.8,
    'lines.markersize'   : 5,
    # Grid — OFF everywhere, no exceptions
    'axes.grid'          : False,
    'grid.linewidth'     : 0.5,
    'grid.alpha'         : 0.3,
    # Figure / save
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
})

print("✓ Global plot settings applied (grid OFF, no downstream overrides needed)")

# =============================================================================
# PDF / PNG style contexts — used ONLY inside save_pdf_png
# =============================================================================

PDF_STYLE = {
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 28,
    'axes.labelsize'     : 28,
    'axes.titlesize'     : 32,
    'xtick.labelsize'    : 26,
    'ytick.labelsize'    : 26,
    'legend.fontsize'    : 22,
    'figure.titlesize'   : 28,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size'   : 6,
    'ytick.major.size'   : 6,
    'xtick.minor.size'   : 3,
    'ytick.minor.size'   : 3,
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.8,
    'axes.grid'          : False,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
}

PNG_STYLE = {
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'   : 'cm',
    'font.size'          : 16,
    'axes.labelsize'     : 22,
    'axes.titlesize'     : 18,
    'xtick.labelsize'    : 20,
    'ytick.labelsize'    : 20,
    'legend.fontsize'    : 18,
    'figure.titlesize'   : 16,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.top'          : True,
    'ytick.right'        : True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'axes.linewidth'     : 1.0,
    'lines.linewidth'    : 1.5,
    'axes.grid'          : False,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
}

print("✓ PDF and PNG style contexts defined")


def save_pdf_png(plot_func, plot_dir, plot_name, title=None):
    """
    Save a plot as both PDF and PNG.

    Parameters
    ----------
    plot_func : callable
        f(ax) — draws onto the provided Axes.
        Do NOT set font sizes or grid inside plot_func.
    plot_dir  : str
    plot_name : str  (no extension)
    title     : str or None  — PNG-only title (PDF has no title)
    """
    with mpl.rc_context(PDF_STYLE):
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        plot_func(ax)
        ax.set_title("")
        ax.grid(False)
        fig.savefig(f"{plot_dir}/{plot_name}.pdf")
        plt.close(fig)

    with mpl.rc_context(PNG_STYLE):
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        plot_func(ax)
        ax.grid(False)
        if title is not None:
            ax.set_title(title, fontweight='bold')
        fig.savefig(f"{plot_dir}/{plot_name}.png")
        plt.close(fig)


print("✓ save_pdf_png defined (plot_func pattern, grid always OFF)")

# =============================================================================
# CELL 1c: Define Parameters
# =============================================================================

user_params = p21c.UserParams(
    HII_DIM=128,
    BOX_LEN=800.0,
    USE_INTERPOLATION_TABLES=True,
    N_THREADS=32
)

z_min = 0.001
z_max = 20.0

# =============================================================================
# MULTI-SEED SETUP
# =============================================================================

RANDOM_SEEDS = list(range(1, 21))   # seeds 1, 2, 3, ..., 20
N_SEEDS      = len(RANDOM_SEEDS)

print(f"\n=== PARAMETER SETUP ===")
print(f"HII_DIM     = {user_params.HII_DIM}")
print(f"BOX_LEN     = {user_params.BOX_LEN:.0f} Mpc")
print(f"z range     = [{z_min}, {z_max}]")
print(f"N_THREADS   = {user_params.N_THREADS}")

print(f"\n=== MULTI-SEED SETUP ===")
print(f"Seeds: {RANDOM_SEEDS}")
print(f"Total realisations: {N_SEEDS}")

print("\n=== DEFAULT COSMOLOGY ===")
print(p21c.CosmoParams())

print("\n=== DEFAULT ASTROPHYSICS ===")
print(p21c.AstroParams())

print("\n=== DEFAULT FLAGS ===")
print(p21c.FlagOptions())

py21cmfast version: 3.4.0
Directory already exists: 18March2026_kSZ2_21cm/plots
All plots will be saved to: /home/swanith/Desktop/Research/Project2/Plots/kSZ_sqr_21cm_lightconev3/18March2026_kSZ2_21cm/plots

=== MULTI-SEED SETUP ===
Seeds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
Total realisations: 60

=== USER PARAMETERS ===
UserParams:
    BOX_LEN                 : 800.0
    DIM                     : 384
    FAST_FCOLL_TABLES       : False
    HII_DIM                 : 128
    HMF                     : 1
    KEEP_3D_VELOCITIES      : False
    MINIMIZE_MEMORY         : False
    NON_CUBIC_FACTOR        : 1.0
    NO_RNG                  : False
    N_THREADS               : 32
    PERTURB_ON_HIGH_RES     : False
    POWER_SPECTRUM          : 0
    USE_2LPT                : True
    USE_FFTW_WISDO

In [ ]:
# =============================================================================
# CELL 2: Run Lightcone Simulations for All Seeds
# Pro caching (native py21cmfast HDF5) + concurrent seeds (ProcessPool/spawn)
# =============================================================================

import time
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

print("\n" + "="*70)
print("RUNNING LIGHTCONE SIMULATIONS FOR kSZ²-21cm ANALYSIS")
print("="*70)

# =============================================================================
# CORE / WORKER ALLOCATION
# =============================================================================
# Total cores available on this PBS node.
# Priority: PBS_NCPUS (set by PBS) → os.cpu_count() → fallback to 32.
N_TOTAL_CORES = int(os.environ.get('PBS_NCPUS', os.cpu_count() or 32))

# Split: prefer 4 workers × 8 threads on 32 cores.
# If more cores available, scale workers up (more concurrency) keeping threads at 8.
# If fewer, drop workers.
N_THREADS_PER_WORKER = 8
N_WORKERS            = max(1, N_TOTAL_CORES // N_THREADS_PER_WORKER)

# Don't spawn more workers than seeds
N_WORKERS = min(N_WORKERS, N_SEEDS)

print(f"\n=== PARALLEL EXECUTION SETUP ===")
print(f"  N_TOTAL_CORES        : {N_TOTAL_CORES}")
print(f"  N_WORKERS            : {N_WORKERS}   (concurrent seeds)")
print(f"  N_THREADS_PER_WORKER : {N_THREADS_PER_WORKER}   (OMP threads each)")
print(f"  Total threads in use : {N_WORKERS * N_THREADS_PER_WORKER}/{N_TOTAL_CORES}")

# =============================================================================
# Astrophysical params summary (defaults)
# =============================================================================
_astro_summary = p21c.AstroParams()
print(f"\nAstrophysical Parameters (defaults):")
print(f"  HII_EFF_FACTOR = {_astro_summary.HII_EFF_FACTOR}")
print(f"  ION_Tvir_MIN   = {_astro_summary.ION_Tvir_MIN:.3f} (log10 K) "
      f"= {10**_astro_summary.ION_Tvir_MIN:.2e} K")
print(f"Redshift range : z = {z_min} → {z_max}")
print(f"Box size       : {user_params.BOX_LEN} Mpc")
print(f"Resolution     : {user_params.HII_DIM}³ cells")


# =============================================================================
# Worker function — runs in a SPAWNED subprocess.
# Must be top-level (picklable). Re-imports py21cmfast in the subprocess.
# Returns the cached HDF5 file path (cheap to pickle); main loads it after.
# =============================================================================
def _run_or_load_seed(seed, seed_cache_dir, z_min, z_max,
                      hii_dim, box_len, n_threads):
    """Run or load one seed. Returns (seed, cache_file_path, sim_time, status)."""
    import os
    import glob
    import time as _time
    import py21cmfast as _p21c

    os.makedirs(seed_cache_dir, exist_ok=True)

    # Build params inside subprocess (CFFI objects don't survive spawn)
    up = _p21c.UserParams(
        HII_DIM=hii_dim,
        BOX_LEN=box_len,
        USE_INTERPOLATION_TABLES=True,
        N_THREADS=n_threads,
    )
    ap = _p21c.AstroParams()

    # -------- CACHE CHECK (native py21cmfast HDF5) ----------------------------
    cached = sorted(glob.glob(os.path.join(seed_cache_dir, "LightCone_*.h5")))
    valid_cached = [(f, os.path.getsize(f) / 1e6)
                    for f in cached if os.path.getsize(f) / 1e6 > 1.0]

    if valid_cached:
        cache_file, size_mb = valid_cached[0]
        # Validate it's loadable by re-running with write=False (uses cache)
        try:
            _ = _p21c.run_lightcone(
                redshift=z_min,
                max_redshift=z_max,
                lightcone_quantities=('brightness_temp', 'density',
                                      'xH_box', 'velocity'),
                user_params=up,
                astro_params=ap,
                random_seed=seed,
                direc=seed_cache_dir,
                write=False,
            )
            return (seed, cache_file, 0.0, "cached")
        except Exception:
            # Cache file present but invalid — fall through to recompute
            pass

    # -------- RUN NEW SIMULATION ---------------------------------------------
    sim_start = _time.time()
    try:
        lc = _p21c.run_lightcone(
            redshift=z_min,
            max_redshift=z_max,
            lightcone_quantities=('brightness_temp', 'density',
                                  'xH_box', 'velocity'),
            user_params=up,
            astro_params=ap,
            random_seed=seed,
            direc=seed_cache_dir,
            write=True,
        )
        # Make sure it's persisted (write=True usually does this, but be explicit)
        try:
            lc.save(direc=seed_cache_dir)
        except Exception:
            pass

        # Find the saved file
        cached = sorted(glob.glob(os.path.join(seed_cache_dir, "LightCone_*.h5")))
        cache_file = cached[0] if cached else None
        return (seed, cache_file, _time.time() - sim_start, "computed")

    except Exception as e:
        return (seed, None, _time.time() - sim_start, f"failed: {e}")


# =============================================================================
# Dispatch — concurrent seeds via ProcessPoolExecutor with spawn context
# =============================================================================
lightcones    = {}
seed_metadata = {}   # {seed: {'cache_file', 'sim_time', 'status'}}

scan_start = time.time()

# spawn context is critical: py21cmfast's CFFI / C globals don't survive fork
mp_ctx = mp.get_context("spawn")

print(f"\n{'='*70}")
print(f"DISPATCHING {N_SEEDS} SEEDS — {N_WORKERS} concurrent workers")
print(f"{'='*70}", flush=True)

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=mp_ctx) as ex:
    futures = {}
    for seed in RANDOM_SEEDS:
        seed_cache_dir = os.path.join(main_cache_dir, f"seed_{seed}")
        fut = ex.submit(
            _run_or_load_seed,
            seed, seed_cache_dir, z_min, z_max,
            user_params.HII_DIM, user_params.BOX_LEN,
            N_THREADS_PER_WORKER,
        )
        futures[fut] = seed

    completed_count = 0
    for fut in as_completed(futures):
        seed = futures[fut]
        try:
            seed_done, cache_file, sim_time, status = fut.result()
        except Exception as e:
            seed_done, cache_file, sim_time, status = seed, None, 0.0, f"crashed: {e}"

        completed_count += 1
        seed_metadata[seed_done] = {
            'cache_file': cache_file,
            'sim_time'  : sim_time,
            'status'    : status,
        }

        if status == "cached":
            msg = f"✓ cached  (instant load)"
        elif status == "computed":
            msg = f"✓ computed in {sim_time/60:.2f} min"
        else:
            msg = f"✗ {status}"

        elapsed = (time.time() - scan_start) / 60
        print(f"  [{completed_count:2d}/{N_SEEDS}] seed {seed_done:3d}: "
              f"{msg}   (elapsed: {elapsed:.1f} min)", flush=True)

print(f"\n{'='*70}")
print(f"✓ ALL WORKERS RETURNED  —  {time.time()-scan_start:.1f} s total")
print(f"{'='*70}")

# =============================================================================
# Load every cached lightcone into the main process
# (single-process load — fast, and gives real LightCone objects downstream)
# =============================================================================
print(f"\n=== LOADING LIGHTCONES INTO MAIN PROCESS ===", flush=True)

astro_params = p21c.AstroParams()   # match the one workers used

for seed in RANDOM_SEEDS:
    meta = seed_metadata.get(seed, {})
    cache_file = meta.get('cache_file')

    if not cache_file or not os.path.exists(cache_file):
        print(f"  ✗ seed {seed}: no cache file — skipping")
        continue

    seed_cache_dir = os.path.dirname(cache_file)
    try:
        lc = p21c.run_lightcone(
            redshift=z_min,
            max_redshift=z_max,
            lightcone_quantities=('brightness_temp', 'density',
                                  'xH_box', 'velocity'),
            user_params=user_params,
            astro_params=astro_params,
            random_seed=seed,
            direc=seed_cache_dir,
            write=False,
        )
        lightcones[seed] = lc

        z_nodes   = lc.node_redshifts[::-1]
        x_e_nodes = 1.0 - lc.global_xH[::-1]
        try:
            z_10 = z_nodes[np.argmin(np.abs(x_e_nodes - 0.1))]
            z_50 = z_nodes[np.argmin(np.abs(x_e_nodes - 0.5))]
            z_90 = z_nodes[np.argmin(np.abs(x_e_nodes - 0.9))]
            reion = (f"z(10%)={z_10:.2f}  z(50%)={z_50:.2f}  "
                     f"z(90%)={z_90:.2f}  Δz={z_10-z_90:.2f}")
        except Exception:
            reion = "(reion stats unavailable)"
        print(f"  ✓ seed {seed:3d} loaded   {reion}")

    except Exception as e:
        print(f"  ✗ seed {seed}: load failed — {e}")

total_time = time.time() - scan_start
print(f"\n{'='*70}")
print(f"✓ CELL 2 COMPLETE   total wall time: {total_time/60:.2f} min")
print(f"  Successful: {len(lightcones)}/{N_SEEDS}")
print(f"  Seeds loaded: {sorted(lightcones.keys())}")
print(f"{'='*70}")


# =============================================================================
# CELL 2b: Plot Lightcone Fields (random seed as sanity check)
# Uses save_pdf_png — no font/grid overrides
# =============================================================================

if len(lightcones) > 0:
    seed_to_plot = int(np.random.choice(list(lightcones.keys())))
    lightcone    = lightcones[seed_to_plot]

    print(f"\n{'='*70}")
    print(f"LIGHTCONE PLOTS (randomly selected seed={seed_to_plot})")
    print(f"{'='*70}")

    fields_to_plot = [
        ('brightness_temp', '21cm Brightness Temperature', 'EoR'),
        ('xH_box',          'Neutral Fraction (xHI)',      'viridis'),
        ('density',         'Overdensity δ',               'magma'),
        ('velocity',        'Line-of-Sight Velocity',      'RdBu_r'),
    ]

    for field_name, field_title, field_cmap in fields_to_plot:
        print(f"  Plotting {field_name}...")

        field_data = np.asarray(getattr(lightcone, field_name))
        mid_slice  = field_data[:, :, field_data.shape[2] // 2]

        def _draw(ax, _slice=mid_slice, _cmap=field_cmap,
                  _title=field_title, _seed=seed_to_plot):
            im = ax.imshow(_slice.T, aspect='auto',
                           cmap=_cmap, origin='lower')
            ax.figure.colorbar(im, ax=ax)
            ax.set_xlabel(r'LoS pixel')
            ax.set_ylabel(r'Transverse pixel')
            ax.text(0.02, 0.98,
                    f'seed={_seed} | '
                    f'HII_EFF_FACTOR={astro_params.HII_EFF_FACTOR:.1f}, '
                    f'ION_Tvir_MIN={astro_params.ION_Tvir_MIN:.2f}',
                    transform=ax.transAxes,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        plot_name = f"{field_name}_lightcone_seed{seed_to_plot}"
        save_pdf_png(
            _draw, plot_dir, plot_name,
            title=f'{field_title} — Lightcone (z={z_max}→{z_min}) '
                  f'[seed={seed_to_plot}]',
            figsize=(12, 5),
        )
        print(f"  ✓ Saved: {plot_name}")

    print("\n✓ LIGHTCONE PLOTTING COMPLETE!")
else:
    print("\n✗ Skipping plots — no lightcones available")


RUNNING LIGHTCONE SIMULATIONS FOR kSZ²-21cm ANALYSIS
Created cache directory: 18March2026_kSZ2_21cm/cache

Astrophysical Parameters:
  HII_EFF_FACTOR = 30.0
  ION_Tvir_MIN   = 4.699 (log10 K) = 5.00e+04 K

Redshift range : z = 0.001 → 20.0
Box size       : 800.0 Mpc
Resolution     : 128³ cells

SEED 1  (1/60)
  No cache found → running simulation...


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# CELL 4: Reionization History + Optical Depth (All Seeds)
# Combines former Cells 3 and 4.
# Produces:
#   - tau_results  (dict, used by downstream kSZ integration)
#   - xe_mean / xHI_mean / tau_mean  (mean curves)
# Plots:
#   - reionization_history_xe   (xe vs z)
#   - reionization_history_xHI  (xHI vs z)
#   - tau_vs_z                   (cumulative τ vs z)
# All plotting via save_pdf_png — no font/grid overrides.
# =============================================================================

print("\n" + "="*70)
print("REIONIZATION HISTORY + OPTICAL DEPTH ANALYSIS")
print("="*70)

if len(lightcones) > 0:

    # =========================================================================
    # PART A: Reionization histories (xe, xHI) — per seed + mean across seeds
    # =========================================================================
    all_z_nodes   = {}
    all_x_e_nodes = {}
    all_xHI_nodes = {}

    for seed, lc in lightcones.items():
        sort_idx              = np.argsort(lc.node_redshifts)
        all_z_nodes[seed]     = lc.node_redshifts[sort_idx]
        all_x_e_nodes[seed]   = (1.0 - lc.global_xH)[sort_idx]
        all_xHI_nodes[seed]   = lc.global_xH[sort_idx]

    # Common redshift grid for averaging — within range shared by all seeds
    z_min_common = max(all_z_nodes[s].min() for s in lightcones)
    z_max_common = min(all_z_nodes[s].max() for s in lightcones)
    z_common_xe  = np.linspace(z_min_common, z_max_common, 500)

    xe_interp = np.array([
        np.interp(z_common_xe, all_z_nodes[s], all_x_e_nodes[s])
        for s in lightcones.keys()
    ])
    xHI_interp = np.array([
        np.interp(z_common_xe, all_z_nodes[s], all_xHI_nodes[s])
        for s in lightcones.keys()
    ])

    xe_mean  = np.mean(xe_interp,  axis=0)
    xe_std   = np.std(xe_interp,   axis=0)
    xHI_mean = np.mean(xHI_interp, axis=0)
    xHI_std  = np.std(xHI_interp,  axis=0)

    # Mean reionization midpoint
    z_xe_half_mean = np.interp(0.5, xe_mean[::-1], z_common_xe[::-1])
    print(f"\nMean z(x_e = 0.5) across {N_SEEDS} seeds: z = {z_xe_half_mean:.2f}")
    for seed in lightcones.keys():
        z_half = np.interp(0.5, all_x_e_nodes[seed], all_z_nodes[seed])
        print(f"  Seed {seed:3d}: z(x_e=0.5) = {z_half:.2f}")

    # Color map for per-seed lines (used in all three plots)
    cmap_seeds = plt.cm.plasma(np.linspace(0.1, 0.9, len(lightcones)))

    # AstroParams instance for annotations
    astro_params = p21c.AstroParams()

    # ---------------------------------------------------------------------
    # PLOT 4a: x_e vs z
    # ---------------------------------------------------------------------
    def _draw_xe(ax):
        for i, seed in enumerate(lightcones.keys()):
            ax.plot(all_z_nodes[seed], all_x_e_nodes[seed],
                    color=cmap_seeds[i], lw=1.0, alpha=0.4)

        ax.fill_between(z_common_xe, xe_mean - xe_std, xe_mean + xe_std,
                        color='darkblue', alpha=0.2, label=r'$\pm 1\sigma$')
        ax.plot(z_common_xe, xe_mean,
                color='darkblue', lw=2.5, label=f'Mean ({N_SEEDS} seeds)')

        ax.axhline(0.5, color='gray', linestyle='--', lw=1, alpha=0.7)
        ax.axvline(z_xe_half_mean, color='gray', linestyle='--', lw=1, alpha=0.7)
        ax.text(z_xe_half_mean, 0.52,
                rf'$z_{{x_e=0.5}} = {z_xe_half_mean:.2f}$',
                ha='right', va='bottom',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        ax.set_xlabel(r'Redshift $z$')
        ax.set_ylabel(r'Ionization Fraction $x_e$')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(loc='best')
        ax.invert_xaxis()

    save_pdf_png(_draw_xe, plot_dir, "reionization_history_xe",
                 title='Reionization History: Ionization Fraction')
    print(f"\n✓ Saved: reionization_history_xe")

    # ---------------------------------------------------------------------
    # PLOT 4b: x_HI vs z
    # ---------------------------------------------------------------------
    def _draw_xHI(ax):
        for i, seed in enumerate(lightcones.keys()):
            ax.plot(all_z_nodes[seed], all_xHI_nodes[seed],
                    color=cmap_seeds[i], lw=1.0, alpha=0.4)

        ax.fill_between(z_common_xe, xHI_mean - xHI_std, xHI_mean + xHI_std,
                        color='darkred', alpha=0.2, label=r'$\pm 1\sigma$')
        ax.plot(z_common_xe, xHI_mean,
                color='darkred', lw=2.5, label=f'Mean ({N_SEEDS} seeds)')

        ax.set_xlabel(r'Redshift $z$')
        ax.set_ylabel(r'Neutral Fraction $x_{\rm HI}$')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(loc='best')
        ax.invert_xaxis()

        ax.text(0.05, 0.95,
                f'HII_EFF_FACTOR = {astro_params.HII_EFF_FACTOR:.1f}\n'
                f'ION_Tvir_MIN = {astro_params.ION_Tvir_MIN:.2f} (log10 K)\n'
                f'N seeds = {N_SEEDS}',
                transform=ax.transAxes,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(_draw_xHI, plot_dir, "reionization_history_xHI",
                 title='Reionization History: Neutral Fraction')
    print(f"✓ Saved: reionization_history_xHI")

    # =========================================================================
    # PART B: Optical depth τ(<z) — per seed, then averaged
    # =========================================================================
    print("\n--- Optical Depth Calculation ---")

    # Physical constants
    c_km_s         = 2.998e5
    h              = 0.6766
    H0             = 100 * h
    Omega_b        = 0.04897468161869667
    Omega_m        = 0.30964144154550644
    rho_crit_p_cm3 = 1.88e-29 * h**2 / (1.67e-24)
    n_H0_cm3       = Omega_b * rho_crit_p_cm3
    sigma_T_cm2    = 6.65e-25
    cm_per_Mpc     = 3.086e24
    n_e0_Mpc3      = n_H0_cm3 * cm_per_Mpc**3
    sigma_T_Mpc2   = sigma_T_cm2 / cm_per_Mpc**2
    prefactor      = n_e0_Mpc3 * sigma_T_Mpc2

    print(f"  n_H0     = {n_H0_cm3:.6e} cm^-3")
    print(f"  σ_T      = {sigma_T_cm2:.6e} cm^2")
    print(f"  Prefactor= {prefactor:.6e} Mpc^-1")

    # ---- Compute τ per seed ----
    tau_results = {}

    for seed, lc in lightcones.items():
        red_axis = lc.lightcone_redshifts
        pos_axis = lc.lightcone_distances

        # Trim to z ≤ z_max
        ind_z    = np.where(red_axis <= z_max)[0]
        red_axis = red_axis[ind_z]
        pos_axis = pos_axis[ind_z]

        # Ionization history (descending → ascending)
        z_nodes_sorted   = lc.node_redshifts[::-1]
        xHI_nodes_sorted = lc.global_xH[::-1]
        x_e_nodes_sorted = 1.0 - xHI_nodes_sorted

        x_e_interp = np.interp(red_axis, z_nodes_sorted, x_e_nodes_sorted)

        ds_Mpc  = np.asarray(np.diff(pos_axis), dtype=np.float64)
        z_mid   = 0.5 * (red_axis[:-1] + red_axis[1:])
        x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])

        dtau      = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds_Mpc
        tau       = np.cumsum(dtau)
        tau_total = tau[-1]

        tau_results[seed] = {
            'red_axis' : red_axis,
            'z_mid'    : z_mid,
            'x_e_mid'  : x_e_mid,
            'ds_Mpc'   : ds_Mpc,
            'tau'      : tau,
            'tau_total': tau_total,
        }
        print(f"  Seed {seed:3d}: τ_total = {tau_total:.6f}")

    tau_totals = np.array([tau_results[s]['tau_total'] for s in lightcones])
    print(f"\n  Mean τ = {tau_totals.mean():.6f} ± {tau_totals.std():.6f}")

    # ---- Stack τ(<z) across seeds — handle potential grid mismatch safely ----
    ref_seed   = next(iter(lightcones))
    ref_z_mid  = tau_results[ref_seed]['z_mid']

    grids_match = all(
        tau_results[s]['z_mid'].shape == ref_z_mid.shape
        and np.allclose(tau_results[s]['z_mid'], ref_z_mid)
        for s in lightcones
    )

    if grids_match:
        z_common_tau = ref_z_mid
        tau_matrix   = np.array([tau_results[s]['tau'] for s in lightcones])
    else:
        # Fallback: interpolate onto a common grid
        print("  ⚠ Seed z_mid grids differ — interpolating onto a common grid")
        z_lo = max(tau_results[s]['z_mid'].min() for s in lightcones)
        z_hi = min(tau_results[s]['z_mid'].max() for s in lightcones)
        z_common_tau = np.linspace(z_lo, z_hi, 1000)
        tau_matrix   = np.array([
            np.interp(z_common_tau,
                      tau_results[s]['z_mid'],
                      tau_results[s]['tau'])
            for s in lightcones
        ])

    tau_mean = np.mean(tau_matrix, axis=0)
    tau_std  = np.std(tau_matrix,  axis=0)

    # ---------------------------------------------------------------------
    # PLOT 4c: τ(<z) vs z
    # ---------------------------------------------------------------------
    def _draw_tau(ax):
        for i, seed in enumerate(lightcones.keys()):
            ax.plot(tau_results[seed]['z_mid'], tau_results[seed]['tau'],
                    color=cmap_seeds[i], lw=1.0, alpha=0.4)

        ax.fill_between(z_common_tau, tau_mean - tau_std, tau_mean + tau_std,
                        color='darkgreen', alpha=0.2, label=r'$\pm 1\sigma$')
        ax.plot(z_common_tau, tau_mean,
                color='darkgreen', lw=2.5, label=f'Mean ({N_SEEDS} seeds)')

        ax.set_xlabel(r'Redshift $z$')
        ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$')
        ax.legend(loc='best')
        ax.invert_xaxis()

        ax.text(0.05, 0.95,
                f'Mean τ = {tau_totals.mean():.4f} ± {tau_totals.std():.4f}\n'
                f'HII_EFF_FACTOR = {astro_params.HII_EFF_FACTOR:.1f}\n'
                f'N seeds = {N_SEEDS}',
                transform=ax.transAxes,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(_draw_tau, plot_dir, "tau_vs_z",
                 title='Cumulative Optical Depth vs Redshift')
    print(f"\n✓ Saved: tau_vs_z")

    print("\n✓ CELL 4 COMPLETE (reionization history + optical depth)")

else:
    print("\n✗ Skipping — no lightcones available")


REIONIZATION HISTORY ANALYSIS
Mean z(x_e = 0.5) across seeds: z = 8.05
  Seed   1: z(x_e=0.5) = 20.13
  Seed   2: z(x_e=0.5) = 20.13
  Seed   3: z(x_e=0.5) = 20.13
  Seed   4: z(x_e=0.5) = 20.13
  Seed   5: z(x_e=0.5) = 20.13
  Seed   6: z(x_e=0.5) = 20.13
  Seed   7: z(x_e=0.5) = 20.13
  Seed   8: z(x_e=0.5) = 20.13
  Seed   9: z(x_e=0.5) = 20.13
  Seed  10: z(x_e=0.5) = 20.13
  Seed  11: z(x_e=0.5) = 20.13
  Seed  12: z(x_e=0.5) = 20.13
  Seed  13: z(x_e=0.5) = 20.13
  Seed  14: z(x_e=0.5) = 20.13
  Seed  15: z(x_e=0.5) = 20.13
  Seed  16: z(x_e=0.5) = 20.13
  Seed  17: z(x_e=0.5) = 20.13
  Seed  18: z(x_e=0.5) = 20.13
  Seed  19: z(x_e=0.5) = 20.13
  Seed  20: z(x_e=0.5) = 20.13
  Seed  21: z(x_e=0.5) = 20.13
  Seed  22: z(x_e=0.5) = 20.13
  Seed  23: z(x_e=0.5) = 20.13
  Seed  24: z(x_e=0.5) = 20.13
  Seed  25: z(x_e=0.5) = 20.13
  Seed  26: z(x_e=0.5) = 20.13
  Seed  27: z(x_e=0.5) = 20.13
  Seed  28: z(x_e=0.5) = 20.13
  Seed  29: z(x_e=0.5) = 20.13
  Seed  30: z(x_e=0.5) = 20.1

In [ ]:
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function (All Seeds)
# kSZ integrand = (1 + δ) × x_e × v_z / c × e^(-τ(z))
# Skips computation for seeds whose Cell 6 kSZ map cache already exists.
# No parallelization needed — per-seed work is just NumPy on 128³ arrays.
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION")
print("="*70)

# Observation redshift — only used to build the Cell-6 cache-skip path
z_obs = 5.0

if len(lightcones) > 0 and len(tau_results) > 0:

    c_Mpc_s = 299792.458 / 3.08567758e19
    print(f"Speed of light: c = {c_Mpc_s:.6e} Mpc/s")

    kSZ_integrands = {}   # {seed: 3D array, or None if skipped}

    for seed, lc in lightcones.items():

        print(f"\n--- Seed {seed} ---")

        # ------------------------------------------------------------------
        # Skip if Cell 6 kSZ map cache already exists — integrand not needed
        # ------------------------------------------------------------------
        map_path = os.path.join(
            main_cache_dir, "kSZ_maps",
            f"kSZ_map_z{z_obs:.1f}_seed{seed}.npy"
        )
        if os.path.exists(map_path):
            print(f"  Cell 6 cache exists → skipping integrand computation")
            kSZ_integrands[seed] = None
            continue

        # ------------------------------------------------------------------
        # Compute integrand
        # ------------------------------------------------------------------
        tr = tau_results[seed]

        red_axis_full = np.asarray(lc.lightcone_redshifts)
        ind_z         = np.where(red_axis_full <= z_max)[0]

        density_1plus = 1 + np.asarray(lc.density[:, :, ind_z])
        x_e_3D        = 1 - np.asarray(lc.xH_box[:, :, ind_z])
        v_los_Mpc_s   = np.asarray(lc.velocity[:, :, ind_z])/67.4

        red_axis_array = np.asarray(tr['red_axis'], dtype=np.float64)
        tau_array      = np.asarray(tr['tau'],      dtype=np.float64)
        z_mid_array    = np.asarray(tr['z_mid'],    dtype=np.float64)

        tau_extended = np.concatenate([[0.0], tau_array])
        z_extended   = np.concatenate([[red_axis_array[0]], z_mid_array])
        tau_at_lc    = np.asarray(
            np.interp(red_axis_array, z_extended, tau_extended),
            dtype=np.float64,
        )

        visibility    = np.exp(-tau_at_lc)
        visibility_3D = visibility[None, None, :]

        kSZ_integrand = (density_1plus * x_e_3D
                         * v_los_Mpc_s / c_Mpc_s
                         * visibility_3D)

        kSZ_integrands[seed] = kSZ_integrand

        print(f"  Shape : {kSZ_integrand.shape}")
        print(f"  Mean  : {kSZ_integrand.mean():.4e}")
        print(f"  Std   : {kSZ_integrand.std():.4e}")
        print(f"  RMS   : {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")

    n_computed = sum(1 for v in kSZ_integrands.values() if v is not None)
    n_skipped  = N_SEEDS - n_computed
    print(f"\n✓ Integrand computed: {n_computed} seeds")
    print(f"  Skipped (Cell 6 cache found): {n_skipped} seeds")

    # =========================================================================
    # PLOT 5a: kSZ Integrand Lightcone (random seed that was actually computed)
    # =========================================================================
    computed_seeds = [s for s, v in kSZ_integrands.items() if v is not None]

    if len(computed_seeds) > 0:
        seed_to_plot  = int(np.random.choice(computed_seeds))
        print(f"\nRandomly selected seed for plot: {seed_to_plot}")

        lc            = lightcones[seed_to_plot]
        kSZ_integrand = kSZ_integrands[seed_to_plot]

        red_axis_full = np.asarray(lc.lightcone_redshifts)
        ind_z         = np.where(red_axis_full <= z_max)[0]

        slice_2D = kSZ_integrand[:, :, kSZ_integrand.shape[2] // 2]
        x_extent = float(np.asarray(lc.lightcone_distances[ind_z].max()))
        y_extent = float(user_params.BOX_LEN)

        # Symmetric color limits at 99th percentile of |integrand|
        vmax = float(np.percentile(np.abs(kSZ_integrand), 99))

        # Distance → redshift mapping for the twin x-axis
        lc_distances_float = np.asarray(lc.lightcone_distances, dtype=np.float64)
        lc_redshifts_float = np.asarray(lc.lightcone_redshifts, dtype=np.float64)

        def _draw_integrand(ax,
                            _slice=slice_2D, _xext=x_extent, _yext=y_extent,
                            _vmax=vmax, _seed=seed_to_plot,
                            _dist=lc_distances_float, _zlc=lc_redshifts_float):
            im = ax.imshow(_slice.T,
                           extent=[0, _xext, 0, _yext],
                           aspect='auto', cmap='seismic', origin='lower',
                           vmin=-_vmax, vmax=_vmax)
            cbar = ax.figure.colorbar(im, ax=ax)
            cbar.set_label(r'kSZ Integrand [dimensionless]')

            ax.set_xlabel('Comoving Distance [Mpc]')
            ax.set_ylabel('Comoving Distance [Mpc]')

            # Twin x-axis: redshift labels at the same comoving-distance ticks
            ax2 = ax.twiny()
            ax2.set_xlim(ax.get_xlim())
            distance_ticks = ax.get_xticks()
            z_ticks = np.interp(distance_ticks, _dist, _zlc)
            ax2.set_xticks(distance_ticks)
            ax2.set_xticklabels([f'{z:.1f}' for z in z_ticks])
            ax2.set_xlabel(r'Redshift $z$')

            ax.text(0.02, 0.98,
                    f'seed={_seed} | '
                    f'HII_EFF_FACTOR={astro_params.HII_EFF_FACTOR:.1f}, '
                    f'ION_Tvir_MIN={astro_params.ION_Tvir_MIN:.2f}',
                    transform=ax.transAxes,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        plot_name = f"kSZ_integrand_with_visibility_seed{seed_to_plot}"
        save_pdf_png(
            _draw_integrand, plot_dir, plot_name,
            title=(r'kSZ Integrand: '
                   r'$(1+\delta)\times x_e\times v_z/c\times e^{-\tau(z)}$'
                   f'  [seed={seed_to_plot}]'),
            figsize=(12, 5),
        )
        print(f"✓ Saved: {plot_name}")

        print("\n✓ kSZ INTEGRAND PLOTTING COMPLETE!")

    else:
        print("\n  All seeds loaded from Cell 6 cache — no integrand plot generated")

else:
    print("\n✗ Skipping — lightcones or tau_results not available")


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION
Speed of light: c = 9.715612e-15 Mpc/s

--- Seed 1 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 2 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 3 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 4 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 5 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 6 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 7 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 8 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 9 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 10 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 11 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 12 ---
  Cell 6 cache exists → skipping integrand computation

--- Seed 13 ---
  Cell 6 cache exists → skipping integr

In [ ]:
# =============================================================================
# CELL 6: Compute Line-of-Sight Integrated kSZ Maps for All Seeds
# kSZ(z_obs=5) = ∫ from z_start to z=5 of
#                [n_e0 σ_T (1/a²) (1+δ) x_e v_z/c e^(-τ) ds]
#
# Cache strategy:
#   1. If kSZ_map_zX.X_seedN.npy exists → load directly (fast path).
#   2. Else if Cell 5 left a usable integrand in memory → integrate it.
#   3. Else → skip (with a clear message telling the user to rerun Cell 5).
#
# No parallelization — per-seed work is one np.sum over a 128² × ~few-hundred
# slice array; NumPy + BLAS handle this in <1 s per seed.
# =============================================================================

import time

print("\n" + "="*70)
print(f"LINE-OF-SIGHT kSZ MAP INTEGRATION AT z_obs = {z_obs:.1f}")
print("="*70)

if len(lightcones) > 0:

    # =========================================================================
    # Cache directory for kSZ maps
    # =========================================================================
    kSZ_maps_dir = os.path.join(main_cache_dir, "kSZ_maps")
    os.makedirs(kSZ_maps_dir, exist_ok=True)
    print(f"kSZ maps directory: {kSZ_maps_dir}")

    # =========================================================================
    # Physical constants (CGS)
    # =========================================================================
    print(f"\n=== PHYSICAL CONSTANTS (CGS) ===")
    c_cm_s      = 3.0e10
    sigma_T_cm2 = 6.6525e-25
    n_e0_cm3    = 2.06e-7
    Mpc_to_cm   = 3.0857e24

    print(f"  c    = {c_cm_s:.2e} cm/s")
    print(f"  σ_T  = {sigma_T_cm2:.4e} cm²")
    print(f"  n_e0 = {n_e0_cm3:.4e} cm⁻³")
    print(f"  1 Mpc= {Mpc_to_cm:.4e} cm")

    prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s
    print(f"  Prefactor n_e0 × σ_T × c = {prefactor_cgs:.4e} s⁻¹")

    print(f"\nz_obs = {z_obs:.1f} (end of reionization)")

    # Is Cell 5's integrand dict available at all? (It might not be if the user
    # jumped straight to Cell 6 after a clean cache load.)
    integrands_in_scope = ('kSZ_integrands' in dir()
                           or 'kSZ_integrands' in globals())

    # =========================================================================
    # Loop over seeds
    # =========================================================================
    kSZ_maps = {}

    for seed, lc in lightcones.items():

        print(f"\n--- Seed {seed} ---")

        map_path = os.path.join(
            kSZ_maps_dir, f"kSZ_map_z{z_obs:.1f}_seed{seed}.npy"
        )

        # ------------------------------------------------------------------
        # CASE 1: Cached kSZ map exists → load and continue
        # ------------------------------------------------------------------
        if os.path.exists(map_path):
            print(f"  Found cached kSZ map → loading")
            kSZ_maps[seed] = np.load(map_path)
            rms = float(np.sqrt(np.mean(kSZ_maps[seed]**2)))
            print(f"  ✓ Loaded | RMS: {rms:.4e}")
            continue

        # ------------------------------------------------------------------
        # CASE 2: Need to compute → check that Cell 5 left an integrand
        # ------------------------------------------------------------------
        integrand_available = (
            integrands_in_scope
            and seed in kSZ_integrands
            and kSZ_integrands[seed] is not None
        )

        if not integrand_available:
            print(f"  ✗ No cached map AND no integrand in memory — skipping")
            print(f"    (delete {kSZ_maps_dir}/kSZ_map_z*_seed{seed}.npy "
                  f"and rerun Cell 5 to recompute)")
            continue

        # ------------------------------------------------------------------
        # CASE 3: Compute kSZ map from integrand
        # ------------------------------------------------------------------
        print(f"  Computing from integrand...")

        tr       = tau_results[seed]
        red_axis = np.asarray(tr['red_axis'], dtype=np.float64)
        ds_Mpc   = np.asarray(tr['ds_Mpc'],   dtype=np.float64)
        z_mid    = np.asarray(tr['z_mid'],    dtype=np.float64)
        ds_cm    = ds_Mpc * Mpc_to_cm

        a                = 1.0 / (1.0 + red_axis)
        a_squared        = a**2
        a_squared_mid    = 0.5 * (a_squared[:-1] + a_squared[1:])
        a_squared_mid_3D = a_squared_mid[None, None, :]

        kSZ_int      = kSZ_integrands[seed]
        kSZ_int_mid  = 0.5 * (kSZ_int[:, :, :-1] + kSZ_int[:, :, 1:])
        kSZ_int_full = ((prefactor_cgs / a_squared_mid_3D)
                        * kSZ_int_mid
                        * (ds_cm / c_cm_s)[None, None, :])

        idx_integrate = np.where(z_mid >= z_obs)[0]
        print(f"  Integration: z = {z_mid[idx_integrate].max():.2f} → "
              f"{z_mid[idx_integrate].min():.2f} "
              f"({len(idx_integrate)} slices)")

        t0      = time.time()
        kSZ_map = np.sum(kSZ_int_full[:, :, idx_integrate], axis=2)
        print(f"  Computed in {time.time()-t0:.2f}s")

        np.save(map_path, kSZ_map)
        print(f"  ✓ Saved to {map_path}")

        kSZ_maps[seed] = kSZ_map
        print(f"  Mean: {kSZ_map.mean():.4e} | "
              f"RMS: {np.sqrt(np.mean(kSZ_map**2)):.4e} | "
              f"Std: {kSZ_map.std():.4e}")

    # =========================================================================
    # Summary
    # =========================================================================
    print(f"\n✓ kSZ MAPS READY: {len(kSZ_maps)}/{N_SEEDS} SEEDS")

    if len(kSZ_maps) > 0:
        rms_all = np.array([np.sqrt(np.mean(kSZ_maps[s]**2))
                            for s in kSZ_maps])
        ref_seed = next(iter(kSZ_maps))
        ref_map  = kSZ_maps[ref_seed]

        print(f"  RMS across seeds : {rms_all.mean():.4e} ± "
              f"{rms_all.std():.4e}")
        print(f"  Map dimensions   : {ref_map.shape[0]} × "
              f"{ref_map.shape[1]} pixels")
        print(f"  Physical size    : {user_params.BOX_LEN:.1f} × "
              f"{user_params.BOX_LEN:.1f} Mpc²")
        print(f"  Pixel size       : "
              f"{user_params.BOX_LEN / ref_map.shape[0]:.2f} Mpc")

else:
    print("\n✗ Skipping — no lightcones available")

print("\n" + "="*70)


LINE-OF-SIGHT kSZ MAP INTEGRATION AT z_obs = 5
Directory exists: 18March2026_kSZ2_21cm/cache/kSZ_maps

=== PHYSICAL CONSTANTS (CGS) ===
c = 3.00e+10 cm/s
σ_T = 6.6525e-25 cm²
n_e0 = 2.0600e-07 cm⁻³
1 Mpc = 3.0857e+24 cm
Prefactor n_e0 × σ_T × c = 4.1112e-21 s⁻¹

z_obs = 5.0 (end of reionization)

--- Seed 1 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 9.1143e-06

--- Seed 2 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 9.8000e-06

--- Seed 3 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 1.0894e-05

--- Seed 4 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 9.6901e-06

--- Seed 5 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 1.1151e-05

--- Seed 6 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 1.0754e-05

--- Seed 7 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 8.8289e-06

--- Seed 8 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 1.0117e-05

--- Seed 9 ---
  Found cached kSZ map → loading
  ✓ Loaded | RMS: 9.9121e-06


In [ ]:
# =============================================================================
# CELL 7: Compute kSZ²-21cm Cross-Correlation Power Spectra (All Seeds)
# Cross-correlate the integrated kSZ² map with 21cm brightness-temperature
# slices at every lightcone node redshift.
#
# Cache strategy (mirrors Cells 5/6):
#   1. If <main_cache_dir>/seed_<N>/cross_corr_seed<N>.npy exists → load.
#   2. Else: get the kSZ map for this seed.
#        a. From memory (kSZ_maps dict, populated by Cell 6).
#        b. From Cell 6 cache file (kSZ_map_zX.X_seedN.npy).
#        c. Otherwise skip the seed with a clear message.
#   3. Compute cross-corr against every node redshift, save .npy.
#
# No parallelization — the per-seed work is FFTs + binned reductions; numpy
# handles this faster than multiprocess overhead would allow.
# =============================================================================

from astropy.cosmology import FlatLambdaCDM

print("\n" + "="*70)
print("COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA")
print("="*70)

if len(lightcones) > 0:

    # =========================================================================
    # Map geometry + k-space grid (identical across seeds)
    # =========================================================================
    npix_side    = user_params.HII_DIM
    box_size_Mpc = float(user_params.BOX_LEN)
    pix_size_Mpc = box_size_Mpc / npix_side
    pix_area     = pix_size_Mpc**2

    print(f"\n=== MAP PROPERTIES ===")
    print(f"  Map size   : {npix_side} × {npix_side} pixels")
    print(f"  Physical   : {box_size_Mpc:.1f} × {box_size_Mpc:.1f} Mpc²")
    print(f"  Pixel size : {pix_size_Mpc:.3f} Mpc/pixel")

    dk        = 2 * np.pi / (npix_side * pix_size_Mpc)
    kx        = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    ky        = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    kgrid     = np.sqrt(kx[:, None]**2 + ky[None, :]**2)
    k_bins    = np.logspace(np.log10(dk), np.log10(kgrid.max() * 0.9), 35)
    k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

    print(f"\n=== k-SPACE GRID ===")
    print(f"  dk      : {dk:.6f} Mpc⁻¹")
    print(f"  k range : [{kgrid.min():.6f}, {kgrid.max():.6f}] Mpc⁻¹")
    print(f"  N bins  : {len(k_centers)}")

    cosmo        = FlatLambdaCDM(H0=67.77, Om0=0.3086)
    kSZ_maps_dir = os.path.join(main_cache_dir, "kSZ_maps")

    # Ensure kSZ_maps dict exists (Cell 6 normally creates it, but we
    # may be jumping straight here on a clean rerun)
    if 'kSZ_maps' not in dir() and 'kSZ_maps' not in globals():
        kSZ_maps = {}

    # =========================================================================
    # Per-seed loop
    # =========================================================================
    cross_corr_results_all = {}

    for seed, lc in lightcones.items():

        seed_idx = list(lightcones.keys()).index(seed)
        print(f"\n{'='*60}")
        print(f"SEED {seed}  ({seed_idx+1}/{N_SEEDS})")
        print(f"{'='*60}")

        # ------------------------------------------------------------------
        # CASE 1: Cross-corr cache exists → load and continue
        # ------------------------------------------------------------------
        seed_cache_dir = os.path.join(main_cache_dir, f"seed_{seed}")
        os.makedirs(seed_cache_dir, exist_ok=True)
        cc_cache = os.path.join(seed_cache_dir, f"cross_corr_seed{seed}.npy")

        if os.path.exists(cc_cache):
            print(f"  Found cached cross-corr → loading")
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded {len(cross_corr_results_all[seed])} redshifts")
            continue

        # ------------------------------------------------------------------
        # CASE 2: Need to compute → fetch the kSZ map
        # ------------------------------------------------------------------
        kSZ_map = None

        if seed in kSZ_maps and kSZ_maps[seed] is not None:
            print(f"  kSZ map found in memory")
            kSZ_map = kSZ_maps[seed]
        else:
            map_path = os.path.join(
                kSZ_maps_dir, f"kSZ_map_z{z_obs:.1f}_seed{seed}.npy"
            )
            if os.path.exists(map_path):
                print(f"  kSZ map not in memory → loading from Cell 6 cache")
                kSZ_map = np.load(map_path)
                kSZ_maps[seed] = kSZ_map
                print(f"  ✓ Loaded kSZ map | "
                      f"RMS: {np.sqrt(np.mean(kSZ_map**2)):.4e}")
            else:
                print(f"  ✗ No kSZ map available for seed {seed} — skipping")
                print(f"    Re-run Cells 5 and 6 to generate the kSZ map")
                continue

        # ------------------------------------------------------------------
        # No high-pass filter applied — k→ℓ conversion (comoving vs
        # angular diameter distance) needs to be settled before filtering
        # ------------------------------------------------------------------
        kSZ_map_filtered = kSZ_map
        print(f"  kSZ map used as-is (no high-pass filter)")
        print(f"  RMS: {kSZ_map.std():.4e}")

        # Square map → FFT → 2D power
        kSZ2_map          = kSZ_map_filtered**2
        kSZ2_map_centered = kSZ2_map - np.mean(kSZ2_map)
        fft_kSZ2_shifted  = np.fft.fftshift(np.fft.fft2(kSZ2_map_centered))
        auto_kSZ2_ps2d    = (np.abs(fft_kSZ2_shifted)**2
                             * pix_area / npix_side**2)
        print(f"  kSZ² RMS: {np.sqrt(np.mean(kSZ2_map**2)):.4e}")

        # ------------------------------------------------------------------
        # Loop over node redshifts → 21cm cross-spectra
        # ------------------------------------------------------------------
        node_redshifts     = np.asarray(lc.node_redshifts[::-1])
        cross_corr_results = {}
        loop_start         = time.time()

        for i, z_21cm in enumerate(node_redshifts):

            lc_redshifts = np.asarray(lc.lightcone_redshifts,
                                      dtype=np.float64)
            idx_closest  = int(np.argmin(np.abs(lc_redshifts - z_21cm)))
            z_actual     = float(lc_redshifts[idx_closest])

            T21_slice          = np.asarray(
                lc.brightness_temp[:, :, idx_closest])
            T21_slice_centered = T21_slice - np.mean(T21_slice)
            fft_T21_shifted    = np.fft.fftshift(
                np.fft.fft2(T21_slice_centered))

            cross_ps2d    = (np.real(np.conj(fft_kSZ2_shifted)
                                     * fft_T21_shifted)
                             * pix_area / npix_side**2)
            auto_T21_ps2d = (np.abs(fft_T21_shifted)**2
                             * pix_area / npix_side**2)

            C_cross_1d            = np.zeros(len(k_centers))
            C_cross_1d_err_sample = np.zeros(len(k_centers))
            C_cross_1d_err_cosmic = np.zeros(len(k_centers))
            C_cross_1d_err_total  = np.zeros(len(k_centers))
            P_kSZ2_1d             = np.zeros(len(k_centers))
            P_T21_1d              = np.zeros(len(k_centers))
            n_modes               = np.zeros(len(k_centers))

            for j in range(len(k_centers)):
                mask  = (kgrid >= k_bins[j]) & (kgrid < k_bins[j+1])
                n_pix = np.sum(mask)

                if n_pix > 0:
                    cross_values             = cross_ps2d[mask]
                    C_cross_1d[j]            = np.mean(cross_values)
                    C_cross_1d_err_sample[j] = (np.std(cross_values)
                                                / np.sqrt(n_pix))
                    P_kSZ2_1d[j]             = np.mean(auto_kSZ2_ps2d[mask])
                    P_T21_1d[j]              = np.mean(auto_T21_ps2d[mask])

                    k_volume   = (box_size_Mpc / (2*np.pi))**3
                    n_modes[j] = (4 * np.pi * k_centers[j]**2 * k_volume
                                  * (k_bins[j+1] - k_bins[j]))

                    if n_modes[j] > 0:
                        C_cross_1d_err_cosmic[j] = (
                            np.sqrt(P_kSZ2_1d[j] * P_T21_1d[j]
                                    + C_cross_1d[j]**2)
                            / np.sqrt(n_modes[j])
                        )
                    else:
                        C_cross_1d_err_cosmic[j] = np.nan

                    C_cross_1d_err_total[j] = np.sqrt(
                        C_cross_1d_err_sample[j]**2
                        + C_cross_1d_err_cosmic[j]**2
                    )
                else:
                    C_cross_1d[j]            = np.nan
                    C_cross_1d_err_sample[j] = np.nan
                    C_cross_1d_err_cosmic[j] = np.nan
                    C_cross_1d_err_total[j]  = np.nan
                    P_kSZ2_1d[j]             = np.nan
                    P_T21_1d[j]              = np.nan

            cross_corr_results[z_21cm] = {
                'k_centers'            : k_centers,
                'C_cross_1d'           : C_cross_1d,
                'C_cross_1d_err_sample': C_cross_1d_err_sample,
                'C_cross_1d_err_cosmic': C_cross_1d_err_cosmic,
                'C_cross_1d_err_total' : C_cross_1d_err_total,
                'n_modes'              : n_modes,
                'P_kSZ2_1d'            : P_kSZ2_1d,
                'P_T21_1d'             : P_T21_1d,
                'z_actual'             : z_actual,
                'idx_closest'          : idx_closest,
                'kSZ2_rms'             : float(np.sqrt(np.mean(kSZ2_map**2))),
                'T21_rms'              : float(np.sqrt(np.mean(T21_slice**2))),
                'T21_mean'             : float(np.mean(T21_slice)),
            }

            if (i+1) % 10 == 0 or i == 0 or i == len(node_redshifts)-1:
                elapsed = time.time() - loop_start
                eta     = (elapsed / (i+1)) * (len(node_redshifts) - (i+1))
                valid   = ~np.isnan(C_cross_1d)
                if np.any(valid):
                    sign_str = ("+" if C_cross_1d[len(C_cross_1d)//2] > 0
                                else "-")
                else:
                    sign_str = "?"
                print(f"  [{i+1:3d}/{len(node_redshifts)}] z={z_21cm:.3f} "
                      f"sign={sign_str} "
                      f"T21_mean={np.mean(T21_slice):.2f} mK | "
                      f"ETA: {eta:.1f}s")

        total_time = time.time() - loop_start
        print(f"\n  ✓ Seed {seed} complete in {total_time/60:.2f} min")

        np.save(cc_cache, cross_corr_results)
        print(f"  ✓ Cached to {cc_cache}")

        cross_corr_results_all[seed] = cross_corr_results

    print(f"\n{'='*70}")
    print(f"✓ CROSS-CORRELATIONS READY: "
          f"{len(cross_corr_results_all)}/{N_SEEDS} SEEDS")

    # =========================================================================
    # Error-budget summary (averaged across all seeds × redshifts × bins)
    # =========================================================================
    print(f"\n=== ERROR BUDGET SUMMARY (averaged across seeds) ===")

    all_sample_err = []
    all_cosmic_err = []
    all_total_err  = []

    for seed, ccr in cross_corr_results_all.items():
        for z_21cm, res in ccr.items():
            C     = res['C_cross_1d']
            valid = ~np.isnan(C) & (C != 0)
            if np.any(valid):
                all_sample_err.extend(
                    (res['C_cross_1d_err_sample'][valid]
                     / np.abs(C[valid])).tolist())
                all_cosmic_err.extend(
                    (res['C_cross_1d_err_cosmic'][valid]
                     / np.abs(C[valid])).tolist())
                all_total_err.extend(
                    (res['C_cross_1d_err_total'][valid]
                     / np.abs(C[valid])).tolist())

    if len(all_sample_err) > 0:
        print(f"  Sample variance (mean): "
              f"{np.nanmean(all_sample_err)*100:.1f}%")
        print(f"  Cosmic variance (mean): "
              f"{np.nanmean(all_cosmic_err)*100:.1f}%")
        print(f"  Total  variance (mean): "
              f"{np.nanmean(all_total_err)*100:.1f}%")

else:
    print("\n✗ Skipping — no lightcones available")

print("\n" + "="*70)


COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA

=== MAP PROPERTIES ===
Map size    : 128 × 128 pixels
Physical    : 800.0 × 800.0 Mpc²
Pixel size  : 6.250 Mpc/pixel

=== k-SPACE GRID ===
dk          : 0.007854 Mpc⁻¹
k range     : [0.000000, 0.710861] Mpc⁻¹

SEED 1  (1/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 2  (2/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 3  (3/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 4  (4/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 5  (5/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 6  (6/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 7  (7/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 8  (8/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 9  (9/60)
  Found cached cross-corr → loading
  ✓ Loaded 155 redshifts

SEED 10  (10/60)
  Found cached cross-corr 

# =============================================================================
# NOT FOR REPORTS
# PLOT: kSZ, kSZ², and 21cm Maps Side-by-Side at Selected Redshifts
# =============================================================================

print(f"\n=== PLOTTING kSZ vs kSZ² vs 21cm MAPS ===")

# Select a few representative redshifts to plot
# Get ionization fraction at each redshift
z_nodes_sorted = lightcone.node_redshifts[::-1]
x_e_nodes = 1.0 - lightcone.global_xH[::-1]

# Find redshifts closest to x_e = 0.2, 0.5, 0.9
target_xe = [0.2, 0.5, 0.9]
selected_z_plot = []

for xe_target in target_xe:
    idx = np.argmin(np.abs(x_e_nodes - xe_target))
    z_sel = z_nodes_sorted[idx]
    # Find closest node redshift from our results
    z_closest = min(cross_corr_results.keys(), key=lambda z: abs(z - z_sel))
    if abs(z_closest - z_sel) < 0.5:  # Reasonable match
        selected_z_plot.append(z_closest)

print(f"Plotting kSZ vs kSZ² vs 21cm for {len(selected_z_plot)} redshifts")

# Create figure: rows = redshifts, cols = [kSZ, kSZ², 21cm]
fig, axes = plt.subplots(len(selected_z_plot), 3, 
                         figsize=(16, 5*len(selected_z_plot)), 
                         constrained_layout=True)

if len(selected_z_plot) == 1:
    axes = axes.reshape(1, -1)

# Get lightcone redshift axis
lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)

for row_idx, z_obs in enumerate(selected_z_plot):
    
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    kSZ_map = np.load(kSZ_map_file)
    
    # Square it
    kSZ2_map = kSZ_map**2
    
    # Find closest lightcone slice to z_obs
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    z_actual = lc_redshifts[idx_closest]
    
    # Extract 21cm brightness temperature slice
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    
    # Get ionization fraction
    x_e = np.interp(z_obs, z_nodes_sorted, x_e_nodes)
    
    # =============================================================================
    # Left panel: kSZ map
    # =============================================================================
    
    ax_kSZ = axes[row_idx, 0]
    
    # Symmetric color scale for kSZ
    vmax_kSZ = np.percentile(np.abs(kSZ_map), 99)
    
    im_kSZ = ax_kSZ.imshow(kSZ_map.T,
                           cmap='seismic',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_kSZ,
                           vmax=vmax_kSZ)
    
    # Colorbar
    cbar_kSZ = plt.colorbar(im_kSZ, ax=ax_kSZ, fraction=0.046, pad=0.04)
    cbar_kSZ.set_label('kSZ (dimensionless)', fontsize=12)
    
    # Labels
    ax_kSZ.set_xlabel('x [Mpc]', fontsize=14)
    ax_kSZ.set_ylabel('y [Mpc]', fontsize=14)
    ax_kSZ.set_title(f'kSZ Map\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                     fontsize=14, fontweight='bold')
    
    # Stats
    rms_kSZ = np.sqrt(np.mean(kSZ_map**2))
    ax_kSZ.text(0.05, 0.95, 
               f'RMS={rms_kSZ:.2e}\nMean={kSZ_map.mean():.2e}',
               transform=ax_kSZ.transAxes, fontsize=11,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Middle panel: kSZ² map
    # =============================================================================
    
    ax_kSZ2 = axes[row_idx, 1]
    
    # Use 'hot' colormap for squared map (all positive)
    vmax_kSZ2 = np.percentile(kSZ2_map, 99)
    
    im_kSZ2 = ax_kSZ2.imshow(kSZ2_map.T,
                             cmap='hot',
                             origin='lower',
                             extent=[0, box_size_Mpc, 0, box_size_Mpc],
                             aspect='equal',
                             vmin=0,
                             vmax=vmax_kSZ2)
    
    # Colorbar
    cbar_kSZ2 = plt.colorbar(im_kSZ2, ax=ax_kSZ2, fraction=0.046, pad=0.04)
    cbar_kSZ2.set_label('kSZ² (dimensionless)', fontsize=12)
    
    # Labels
    ax_kSZ2.set_xlabel('x [Mpc]', fontsize=14)
    ax_kSZ2.set_ylabel('y [Mpc]', fontsize=14)
    ax_kSZ2.set_title(f'kSZ² Map\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                      fontsize=14, fontweight='bold')
    
    # Stats
    rms_kSZ2 = np.sqrt(np.mean(kSZ2_map**2))
    ax_kSZ2.text(0.05, 0.95, 
                f'RMS={rms_kSZ2:.2e}\nMean={kSZ2_map.mean():.2e}',
                transform=ax_kSZ2.transAxes, fontsize=11,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Right panel: 21cm brightness temperature map
    # =============================================================================
    
    ax_T21 = axes[row_idx, 2]
    
    # Use 'RdBu_r' or 'coolwarm' for 21cm (can be positive or negative)
    vmax_T21 = np.percentile(np.abs(T21_slice), 99)
    
    im_T21 = ax_T21.imshow(T21_slice.T,
                           cmap='RdBu_r',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_T21,
                           vmax=vmax_T21)
    
    # Colorbar
    cbar_T21 = plt.colorbar(im_T21, ax=ax_T21, fraction=0.046, pad=0.04)
    cbar_T21.set_label('21cm Brightness Temp [mK]', fontsize=12)
    
    # Labels
    ax_T21.set_xlabel('x [Mpc]', fontsize=14)
    ax_T21.set_ylabel('y [Mpc]', fontsize=14)
    ax_T21.set_title(f'21cm Map\nz={z_actual:.2f}, $x_e$={x_e:.2f}', 
                     fontsize=14, fontweight='bold')
    
    # Stats
    rms_T21 = np.sqrt(np.mean(T21_slice**2))
    ax_T21.text(0.05, 0.95, 
               f'RMS={rms_T21:.2f} mK\nMean={T21_slice.mean():.2f} mK',
               transform=ax_T21.transAxes, fontsize=11,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Overall title
fig.suptitle('kSZ, kSZ², and 21cm Maps at Different Reionization Epochs', 
            fontsize=20, fontweight='bold')

# Save
plot_name = "kSZ_kSZ2_21cm_maps_comparison"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ kSZ vs kSZ² vs 21cm COMPARISON PLOTTING COMPLETE!")

# =============================================================================
# NOT FOR REPORTS
# PLOT 1: 2D FFT Maps (k-space) - kSZ² and 21cm Side-by-Side
# =============================================================================

print(f"\n=== PLOTTING 2D FFT MAPS IN k-SPACE ===")

# Select a few representative redshifts to plot
# Get ionization fraction at each redshift
z_nodes_sorted = lightcone.node_redshifts[::-1]
x_e_nodes = 1.0 - lightcone.global_xH[::-1]

# Find redshifts closest to x_e = 0.2, 0.5, 0.9
target_xe = [0.2, 0.5, 0.9]
selected_z_fft = []

for xe_target in target_xe:
    idx = np.argmin(np.abs(x_e_nodes - xe_target))
    z_sel = z_nodes_sorted[idx]
    # Find closest node redshift from our results
    z_closest = min(cross_corr_results.keys(), key=lambda z: abs(z - z_sel))
    if abs(z_closest - z_sel) < 0.5:  # Reasonable match
        selected_z_fft.append(z_closest)

print(f"Plotting 2D FFT for {len(selected_z_fft)} redshifts")

# Create figure: rows = redshifts, cols = [kSZ² FFT, 21cm FFT]
fig, axes = plt.subplots(len(selected_z_fft), 2, 
                         figsize=(14, 6*len(selected_z_fft)), 
                         constrained_layout=True)

if len(selected_z_fft) == 1:
    axes = axes.reshape(1, -1)

# Get lightcone redshift axis
lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)

for row_idx, z_obs in enumerate(selected_z_fft):
    
    # Recompute FFTs for this redshift
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    kSZ_map = np.load(kSZ_map_file)
    kSZ2_map = kSZ_map**2
    kSZ2_map_centered = kSZ2_map - np.mean(kSZ2_map)
    
    # Get 21cm slice
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    T21_slice_centered = T21_slice - np.mean(T21_slice)
    
    # Compute FFTs
    fft_kSZ2 = np.fft.fft2(kSZ2_map_centered)
    fft_kSZ2_shifted = np.fft.fftshift(fft_kSZ2)
    
    fft_T21 = np.fft.fft2(T21_slice_centered)
    fft_T21_shifted = np.fft.fftshift(fft_T21)
    
    # Get ionization fraction
    x_e = np.interp(z_obs, z_nodes_sorted, x_e_nodes)
    
    # k-space extent
    k_max = kgrid.max()
    
    # =============================================================================
    # Left panel: kSZ² FFT
    # =============================================================================
    
    ax_fft_kSZ2 = axes[row_idx, 0]
    
    # Plot log10 of power
    power_kSZ2 = np.abs(fft_kSZ2_shifted)**2
    power_kSZ2_log = np.log10(power_kSZ2 + 1e-20)  # Add small value to avoid log(0)
    
    im_fft_kSZ2 = ax_fft_kSZ2.imshow(power_kSZ2_log.T,
                                      cmap='viridis',
                                      origin='lower',
                                      extent=[-k_max, k_max, -k_max, k_max],
                                      aspect='equal')
    
    # Colorbar
    cbar_fft_kSZ2 = plt.colorbar(im_fft_kSZ2, ax=ax_fft_kSZ2, fraction=0.046, pad=0.04)
    cbar_fft_kSZ2.set_label(r'log$_{10}$(Power)', fontsize=12)
    
    # Labels
    ax_fft_kSZ2.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_kSZ2.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_kSZ2.set_title(f'kSZ² Power (k-space)\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                          fontsize=14, fontweight='bold')
    
    # Add circle at k = 0.1 Mpc^-1 for reference
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_kSZ2.add_patch(circle)
    
    # =============================================================================
    # Right panel: 21cm FFT
    # =============================================================================
    
    ax_fft_T21 = axes[row_idx, 1]
    
    # Plot log10 of power
    power_T21 = np.abs(fft_T21_shifted)**2
    power_T21_log = np.log10(power_T21 + 1e-20)
    
    im_fft_T21 = ax_fft_T21.imshow(power_T21_log.T,
                                    cmap='viridis',
                                    origin='lower',
                                    extent=[-k_max, k_max, -k_max, k_max],
                                    aspect='equal')
    
    # Colorbar
    cbar_fft_T21 = plt.colorbar(im_fft_T21, ax=ax_fft_T21, fraction=0.046, pad=0.04)
    cbar_fft_T21.set_label(r'log$_{10}$(Power)', fontsize=12)
    
    # Labels
    ax_fft_T21.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_T21.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_T21.set_title(f'21cm Power (k-space)\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                         fontsize=14, fontweight='bold')
    
    # Add circle at k = 0.1 Mpc^-1 for reference
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_T21.add_patch(circle)

# Overall title
fig.suptitle('2D Power Spectra in k-space', 
            fontsize=20, fontweight='bold')

# Save
plot_name = "2D_FFT_kspace_kSZ2_21cm"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 2: Cross-Power and Auto-Power Spectra vs k
# =============================================================================

print(f"\n=== PLOTTING POWER SPECTRA vs k ===")

# Plot for the same selected redshifts
fig, axes = plt.subplots(len(selected_z_fft), 1, 
                         figsize=(10, 6*len(selected_z_fft)), 
                         constrained_layout=True)

if len(selected_z_fft) == 1:
    axes = [axes]

for row_idx, z_obs in enumerate(selected_z_fft):
    
    if z_obs not in cross_corr_results:
        continue
    
    results = cross_corr_results[z_obs]
    k_centers = results['k_centers']
    C_cross = results['C_cross_1d']
    P_kSZ2 = results['P_kSZ2_1d']
    P_T21 = results['P_T21_1d']
    
    # Get ionization fraction
    x_e = np.interp(z_obs, z_nodes_sorted, x_e_nodes)
    
    ax = axes[row_idx]
    
    # Filter valid points
    valid_cross = ~np.isnan(C_cross) & np.isfinite(C_cross)
    valid_kSZ2 = ~np.isnan(P_kSZ2) & (P_kSZ2 > 0)
    valid_T21 = ~np.isnan(P_T21) & (P_T21 > 0)
    
    # Plot auto-power spectra
    ax.loglog(k_centers[valid_kSZ2], P_kSZ2[valid_kSZ2], 
             'o-', color='red', linewidth=2, markersize=4,
             label='kSZ² Auto-Power', alpha=0.8)
    
    ax.loglog(k_centers[valid_T21], P_T21[valid_T21], 
             's-', color='blue', linewidth=2, markersize=4,
             label='21cm Auto-Power', alpha=0.8)
    
    # Plot cross-power (can be negative, so plot absolute value)
    # Use different markers for positive/negative
    positive_mask = valid_cross & (C_cross > 0)
    negative_mask = valid_cross & (C_cross < 0)
    
    if np.any(positive_mask):
        ax.loglog(k_centers[positive_mask], np.abs(C_cross[positive_mask]), 
                 '^-', color='green', linewidth=2.5, markersize=6,
                 label='|Cross-Power| (positive)', alpha=0.9)
    
    if np.any(negative_mask):
        ax.loglog(k_centers[negative_mask], np.abs(C_cross[negative_mask]), 
                 'v--', color='purple', linewidth=2.5, markersize=6,
                 label='|Cross-Power| (negative)', alpha=0.9)
    
    ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=16)
    ax.set_ylabel(r'Power [Mpc$^2$]', fontsize=16)
    ax.set_title(f'Power Spectra: z={z_obs:.2f}, $x_e$={x_e:.2f}', 
                fontsize=16, fontweight='bold')
    ax.legend(fontsize=12, loc='best')
    #ax.grid(True, alpha=0.3)
    
    # Add text showing sign of cross-power
    if np.any(valid_cross):
        mean_sign = "positive" if np.mean(C_cross[valid_cross]) > 0 else "negative"
        ax.text(0.05, 0.95, f'Cross-power: {mean_sign}',
               transform=ax.transAxes, fontsize=12, fontweight='bold',
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Overall title
fig.suptitle('Power Spectra vs k (kSZ², 21cm, and Cross-Power)', 
            fontsize=18, fontweight='bold')

# Save
plot_name = "power_spectra_vs_k_comparison"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 3: Cross-Power Sign Evolution vs k at Different Redshifts
# =============================================================================

print(f"\n=== PLOTTING CROSS-POWER SIGN EVOLUTION ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

# Select more redshifts for this plot
z_sample = sorted(cross_corr_results.keys())[::15]  # Every 15th redshift

cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=min(z_sample), vmax=max(z_sample))

for z_obs in z_sample:
    results = cross_corr_results[z_obs]
    k_centers = results['k_centers']
    C_cross = results['C_cross_1d']
    
    valid = ~np.isnan(C_cross) & np.isfinite(C_cross)
    
    if np.sum(valid) > 5:
        color = cmap(norm(z_obs))
        
        # Plot with sign preserved (use symlog or just regular plot)
        ax.plot(k_centers[valid], C_cross[valid], 
               color=color, linewidth=2, alpha=0.7,
               marker='o', markersize=3)

ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=18)
ax.set_ylabel(r'Cross-Power [Mpc$^2$]', fontsize=18)
ax.set_xscale('log')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
#ax.grid(True, alpha=0.3)

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'Redshift $z$', fontsize=16)

ax.set_title('kSZ²-21cm Cross-Power vs k (Sign Evolution)', 
            fontsize=18, fontweight='bold')

# Save
plot_name = "cross_power_vs_k_sign_evolution"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ ALL DIAGNOSTIC PLOTTING COMPLETE!")

In [ ]:
# =============================================================================
# CELL 8a: Visualize kSZ²-21cm Cross-Correlation Power Spectra (Random Seed)
# Convert to ℓ-space and create five plots for one randomly chosen realisation.
# All plotting via save_pdf_png — no font/grid overrides anywhere.
# =============================================================================

from astropy.cosmology import FlatLambdaCDM

print("\n" + "="*70)
print("VISUALIZING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA")
print("="*70)

# Subdirectory for these final plots
plot_dir_final = os.path.join(plot_dir, "plot_final_cell")
os.makedirs(plot_dir_final, exist_ok=True)
plot_dir_save = plot_dir_final

# =============================================================================
# Load cross_corr_results_all from cache if not already in memory
# =============================================================================
if ('cross_corr_results_all' not in dir()
        and 'cross_corr_results_all' not in globals()) \
   or len(cross_corr_results_all) == 0:

    print("cross_corr_results_all not in memory → loading from Cell 7 cache")
    cross_corr_results_all = {}
    for seed in RANDOM_SEEDS:
        cc_cache = os.path.join(
            main_cache_dir, f"seed_{seed}", f"cross_corr_seed{seed}.npy"
        )
        if os.path.exists(cc_cache):
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded seed {seed} "
                  f"({len(cross_corr_results_all[seed])} redshifts)")
        else:
            print(f"  ✗ No cache found for seed {seed}")
    print(f"  Loaded {len(cross_corr_results_all)}/{N_SEEDS} seeds")
else:
    print(f"cross_corr_results_all already in memory "
          f"({len(cross_corr_results_all)} seeds)")


if len(cross_corr_results_all) > 0:

    # Pick one random seed
    seed_to_plot = int(np.random.choice(list(cross_corr_results_all.keys())))
    print(f"\nRandomly selected seed for plots: {seed_to_plot}")

    cross_corr_results = cross_corr_results_all[seed_to_plot]
    lc                 = lightcones[seed_to_plot]
    slabel             = f"seed{seed_to_plot}"

    # =========================================================================
    # Convert k-space → ℓ-space (per redshift)
    # =========================================================================
    print(f"\n=== CONVERTING TO ℓ-SPACE WITH ERROR PROPAGATION ===")

    T_CMB_0_K = 2.725
    cosmo     = FlatLambdaCDM(H0=67.77, Om0=0.3086)

    cross_corr_ell_results = {}

    for z_node in sorted(cross_corr_results.keys()):
        results          = cross_corr_results[z_node]
        D_A_Mpc          = float(cosmo.angular_diameter_distance(z_node).value)
        chi_comoving_Mpc = float(cosmo.comoving_distance(z_node).value)
        T_CMB_z_uK       = T_CMB_0_K * 1e6

        k_centers  = results['k_centers']
        ell_from_k = k_centers * chi_comoving_Mpc / 0.67

        C_cross_ell            = results['C_cross_1d']            * 0.67**2 / D_A_Mpc**2
        C_cross_ell_err_sample = results['C_cross_1d_err_sample'] * 0.67**2 / D_A_Mpc**2
        C_cross_ell_err_cosmic = results['C_cross_1d_err_cosmic'] * 0.67**2 / D_A_Mpc**2
        C_cross_ell_err_total  = results['C_cross_1d_err_total']  * 0.67**2 / D_A_Mpc**2

        D_cross_ell            = ell_from_k * (ell_from_k + 1) * C_cross_ell            / (2 * np.pi)
        D_cross_ell_err_sample = ell_from_k * (ell_from_k + 1) * C_cross_ell_err_sample / (2 * np.pi)
        D_cross_ell_err_cosmic = ell_from_k * (ell_from_k + 1) * C_cross_ell_err_cosmic / (2 * np.pi)
        D_cross_ell_err_total  = ell_from_k * (ell_from_k + 1) * C_cross_ell_err_total  / (2 * np.pi)

        D_cross_ell_uK_mK            = D_cross_ell            * T_CMB_z_uK**2
        D_cross_ell_uK_mK_err_sample = D_cross_ell_err_sample * T_CMB_z_uK**2
        D_cross_ell_uK_mK_err_cosmic = D_cross_ell_err_cosmic * T_CMB_z_uK**2
        D_cross_ell_uK_mK_err_total  = D_cross_ell_err_total  * T_CMB_z_uK**2

        P_kSZ2_ell = results['P_kSZ2_1d'] * 0.67**2 / D_A_Mpc**2
        P_T21_ell  = results['P_T21_1d']  * 0.67**2 / D_A_Mpc**2
        with np.errstate(divide='ignore', invalid='ignore'):
            r_cross = C_cross_ell / np.sqrt(P_kSZ2_ell * P_T21_ell)

        cross_corr_ell_results[z_node] = {
            'ell_from_k'                  : ell_from_k,
            'D_cross_ell_uK_mK'           : D_cross_ell_uK_mK,
            'D_cross_ell_uK_mK_err_sample': D_cross_ell_uK_mK_err_sample,
            'D_cross_ell_uK_mK_err_cosmic': D_cross_ell_uK_mK_err_cosmic,
            'D_cross_ell_uK_mK_err_total' : D_cross_ell_uK_mK_err_total,
            'D_cross_ell_dimensionless'   : D_cross_ell,
            'r_cross'                     : r_cross,
            'D_A_Mpc'                     : D_A_Mpc,
            'T_CMB_z_uK'                  : T_CMB_z_uK,
        }

    print(f"Converted {len(cross_corr_ell_results)} redshifts to ℓ-space")

    z_values = np.array(sorted(cross_corr_ell_results.keys()))
    cmap     = mpl.cm.rainbow
    norm     = mpl.colors.Normalize(vmin=z_values.min(), vmax=z_values.max())

    # =========================================================================
    # PLOT 1: Rainbow D_ℓ vs ℓ
    # =========================================================================
    print(f"\n=== PLOT 1: Rainbow D_ℓ vs ℓ ===")

    def _draw_p1(ax):
        for z_node in z_values[::2]:
            results   = cross_corr_ell_results[z_node]
            ell       = results['ell_from_k']
            D_ell     = results['D_cross_ell_uK_mK']
            D_ell_err = results['D_cross_ell_uK_mK_err_total']
            valid = (~np.isnan(D_ell) & np.isfinite(D_ell)
                     & (ell > 10) & ~np.isnan(D_ell_err))
            if np.sum(valid) > 5:
                color = cmap(norm(z_node))
                ax.plot(ell[valid], D_ell[valid],
                        color=color, lw=1.5, alpha=0.8)
                ax.fill_between(ell[valid],
                                D_ell[valid] - D_ell_err[valid],
                                D_ell[valid] + D_ell_err[valid],
                                color=color, alpha=0.15)

        ax.set_xlabel(r'Multipole $\ell$')
        ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)')
        ax.set_xscale('log')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)

        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        ax.figure.colorbar(sm, ax=ax, pad=0.02).set_label(r'Redshift $z$')

        ax.text(0.02, 0.02, f'seed={seed_to_plot}',
                transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_p1, plot_dir_save,
        f"kSZ2_21cm_cross_Dl_vs_ell_rainbow_{slabel}",
        title=r'kSZ$^2$-21cm Cross-Power $D_\ell$ vs Redshift',
        figsize=(12, 8),
    )
    print(f"✓ Saved: kSZ2_21cm_cross_Dl_vs_ell_rainbow_{slabel}")

    # =========================================================================
    # PLOT 2: Correlation Coefficient r vs ℓ
    # =========================================================================
    print(f"\n=== PLOT 2: Correlation Coefficient r vs ℓ ===")

    def _draw_p2(ax):
        for z_node in z_values:
            results = cross_corr_ell_results[z_node]
            ell     = results['ell_from_k']
            r       = results['r_cross']
            valid = (~np.isnan(r) & np.isfinite(r)
                     & (ell > 10) & (np.abs(r) < 1.5))
            if np.sum(valid) > 5:
                ax.plot(ell[valid], r[valid],
                        color=cmap(norm(z_node)), lw=1.5, alpha=0.7)

        ax.set_xlabel(r'Multipole $\ell$')
        ax.set_ylabel(r'Correlation Coefficient $r$')
        ax.set_xscale('log')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.set_ylim(-1.2, 1.2)

        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        ax.figure.colorbar(sm, ax=ax, pad=0.02).set_label(r'Redshift $z$')

        ax.text(0.02, 0.02, f'seed={seed_to_plot}',
                transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_p2, plot_dir_save,
        f"kSZ2_21cm_cross_r_vs_ell_rainbow_{slabel}",
        title=r'kSZ$^2$-21cm Correlation Coefficient vs Redshift',
        figsize=(12, 8),
    )
    print(f"✓ Saved: kSZ2_21cm_cross_r_vs_ell_rainbow_{slabel}")

    # =========================================================================
    # PLOT 3: Selected x_e with error bars
    # =========================================================================
    print(f"\n=== PLOT 3: Selected ionization fractions ===")

    z_nodes_sorted = lc.node_redshifts[::-1]
    x_e_nodes      = 1.0 - lc.global_xH[::-1]
    target_xe      = [0.2, 0.5, 0.9]
    selected_z = [z_nodes_sorted[np.argmin(np.abs(x_e_nodes - xe))]
                  for xe in target_xe]
    selected_xe = [x_e_nodes[np.argmin(np.abs(x_e_nodes - xe))]
                   for xe in target_xe]

    print("  Selected redshifts:")
    for z, xe in zip(selected_z, selected_xe):
        print(f"    z={z:.2f}, x_e={xe:.3f}")

    colors_selected = ['blue', 'green', 'red']

    def _draw_p3(ax):
        for i, (z_node, xe) in enumerate(zip(selected_z, selected_xe)):
            z_closest = min(cross_corr_ell_results.keys(),
                            key=lambda z: abs(z - z_node))
            if abs(z_closest - z_node) > 0.5:
                continue
            results          = cross_corr_ell_results[z_closest]
            ell              = results['ell_from_k']
            D_ell            = results['D_cross_ell_uK_mK']
            D_ell_err_total  = results['D_cross_ell_uK_mK_err_total']
            D_ell_err_sample = results['D_cross_ell_uK_mK_err_sample']
            valid = (~np.isnan(D_ell) & np.isfinite(D_ell)
                     & (ell > 10) & ~np.isnan(D_ell_err_total))
            if np.sum(valid) > 5:
                ax.errorbar(ell[valid], D_ell[valid],
                            yerr=D_ell_err_total[valid],
                            color=colors_selected[i], lw=2.5, alpha=0.8,
                            marker='o', markersize=5,
                            capsize=3, capthick=1.5,
                            label=f'z={z_closest:.1f} ($x_e$={xe:.2f})',
                            errorevery=3)
                ax.fill_between(ell[valid],
                                D_ell[valid] - D_ell_err_sample[valid],
                                D_ell[valid] + D_ell_err_sample[valid],
                                color=colors_selected[i], alpha=0.15)

        ax.set_xlabel(r'Multipole $\ell$')
        ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)')
        ax.set_xscale('log')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.legend(loc='best', framealpha=0.9)

        ax.text(0.02, 0.02, f'seed={seed_to_plot}',
                transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_p3, plot_dir_save,
        f"kSZ2_21cm_cross_Dl_selected_xe_{slabel}",
        title=r'kSZ$^2$-21cm Cross-Power at Key Ionization Fractions',
    )
    print(f"✓ Saved: kSZ2_21cm_cross_Dl_selected_xe_{slabel}")

    # =========================================================================
    # PLOT 4: D_ℓ vs z at fixed ℓ
    # =========================================================================
    print(f"\n=== PLOT 4: D_ℓ evolution at fixed ℓ ===")

    ell_targets = [500, 1000, 3000]
    colors_ell  = ['darkblue', 'darkgreen', 'darkred']

    def _draw_p4(ax):
        for i, ell_target in enumerate(ell_targets):
            z_plot, D_plot, D_err_plot = [], [], []
            for z_node in sorted(cross_corr_ell_results.keys()):
                results = cross_corr_ell_results[z_node]
                ell     = results['ell_from_k']
                D_ell   = results['D_cross_ell_uK_mK']
                D_err   = results['D_cross_ell_uK_mK_err_total']
                idx     = np.argmin(np.abs(ell - ell_target))
                if np.isfinite(D_ell[idx]) and np.isfinite(D_err[idx]):
                    z_plot.append(z_node)
                    D_plot.append(D_ell[idx])
                    D_err_plot.append(D_err[idx])

            if len(z_plot) > 0:
                z_plot     = np.array(z_plot)
                D_plot     = np.array(D_plot)
                D_err_plot = np.array(D_err_plot)
                ax.errorbar(z_plot, D_plot, yerr=D_err_plot,
                            color=colors_ell[i], lw=2.5, alpha=0.8,
                            marker='o', markersize=5,
                            capsize=4, capthick=1.5,
                            label=f'$\\ell$={ell_target}', errorevery=2)
                ax.fill_between(z_plot,
                                D_plot - D_err_plot,
                                D_plot + D_err_plot,
                                color=colors_ell[i], alpha=0.15)

        ax.set_xlabel(r'Redshift $z$')
        ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.legend(loc='best', framealpha=0.9)
        ax.invert_xaxis()

        ax.text(0.05, 0.95, f'seed={seed_to_plot}',
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_p4, plot_dir_save,
        f"kSZ2_21cm_cross_Dl_vs_z_fixed_ell_{slabel}",
        title=r'kSZ$^2$-21cm Cross-Power Evolution at Fixed $\ell$',
    )
    print(f"✓ Saved: kSZ2_21cm_cross_Dl_vs_z_fixed_ell_{slabel}")

    # =========================================================================
    # PLOT 5: Error Budget (two stacked panels, sharex)
    # =========================================================================
    print(f"\n=== PLOT 5: Error budget ===")

    z_example = min(cross_corr_ell_results.keys(),
                    key=lambda z: abs(z - selected_z[1]))
    results          = cross_corr_ell_results[z_example]
    ell_p5           = results['ell_from_k']
    D_ell_p5         = results['D_cross_ell_uK_mK']
    D_err_sample_p5  = results['D_cross_ell_uK_mK_err_sample']
    D_err_cosmic_p5  = results['D_cross_ell_uK_mK_err_cosmic']
    D_err_total_p5   = results['D_cross_ell_uK_mK_err_total']
    valid_p5 = ~np.isnan(D_ell_p5) & (ell_p5 > 10) & (D_ell_p5 != 0)

    def _draw_p5(axes):
        ax1, ax2 = axes

        # Top panel: D_ℓ with total error
        ax1.errorbar(ell_p5[valid_p5], D_ell_p5[valid_p5],
                     yerr=D_err_total_p5[valid_p5],
                     fmt='o-', color='darkblue', lw=2, markersize=4,
                     capsize=3, alpha=0.8,
                     label='Cross-power ± total error')
        ax1.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax1.set_ylabel(r'$D_\ell$ [μK$^2$·mK]')
        ax1.set_xscale('log')
        ax1.legend()

        # Bottom panel: fractional error decomposition
        frac_sample = (D_err_sample_p5[valid_p5]
                       / np.abs(D_ell_p5[valid_p5]) * 100)
        frac_cosmic = (D_err_cosmic_p5[valid_p5]
                       / np.abs(D_ell_p5[valid_p5]) * 100)
        frac_total  = (D_err_total_p5[valid_p5]
                       / np.abs(D_ell_p5[valid_p5]) * 100)

        ax2.plot(ell_p5[valid_p5], frac_sample, 'o-',
                 color='blue', lw=2, markersize=4, alpha=0.7,
                 label='Sample variance')
        ax2.plot(ell_p5[valid_p5], frac_cosmic, 's-',
                 color='red', lw=2, markersize=4, alpha=0.7,
                 label='Cosmic variance')
        ax2.plot(ell_p5[valid_p5], frac_total, '^-',
                 color='black', lw=2.5, markersize=5, alpha=0.8,
                 label='Total')
        ax2.set_xlabel(r'Multipole $\ell$')
        ax2.set_ylabel('Fractional Error (%)')
        ax2.set_xscale('log')
        ax2.set_yscale('log')
        ax2.legend()

    save_pdf_png(
        _draw_p5, plot_dir_save,
        f"kSZ2_21cm_cross_error_budget_{slabel}",
        title=f'Error Budget (z={z_example:.2f}, seed={seed_to_plot})',
        figsize=(10, 10),
        n_rows=2,
    )
    print(f"✓ Saved: kSZ2_21cm_cross_error_budget_{slabel}")

    print(f"\n✓ ALL PLOTS COMPLETE (seed={seed_to_plot})")

else:
    print("\n✗ Skipping — no cross_corr_results_all available")

print("\n" + "="*70)


VISUALIZING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA
cross_corr_results_all already in memory (60 seeds)

Randomly selected seed for plots: 37

=== CONVERTING TO ℓ-SPACE WITH ERROR PROPAGATION ===
Converted 155 redshifts to ℓ-space

=== PLOT 1: Rainbow D_ℓ vs ℓ ===
✓ Saved: kSZ2_21cm_cross_Dl_vs_ell_rainbow_seed37

=== PLOT 2: Correlation Coefficient r vs ℓ ===
✓ Saved: kSZ2_21cm_cross_r_vs_ell_rainbow_seed37

=== PLOT 3: Selected ionization fractions ===
  Selected redshifts:
    z=9.78, x_e=0.205
    z=8.02, x_e=0.508
    z=6.40, x_e=0.901
✓ Saved: kSZ2_21cm_cross_Dl_selected_xe_seed37

=== PLOT 4: D_ℓ evolution at fixed ℓ ===
✓ Saved: kSZ2_21cm_cross_Dl_vs_z_fixed_ell_seed37

=== PLOT 5: Error budget ===
✓ Saved: kSZ2_21cm_cross_error_budget_seed37

✓ ALL PLOTS COMPLETE (seed=37)



import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo

# Redshift range
z = np.linspace(0.1, 20, 300)

# Distances
chi_comoving = cosmo.comoving_distance(z).value          # Mpc
D_A = cosmo.angular_diameter_distance(z).value           # Mpc

# Ratio
ratio = chi_comoving/ D_A

# Plot
fig, axs = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

# Top panel: distances
axs[0].plot(z, chi_comoving, lw=2, label=r'Comoving distance $\chi(z)$')
axs[0].plot(z, D_A, lw=2, label=r'Angular diameter distance $D_A(z)$')
axs[0].set_ylabel('Distance [Mpc]')
axs[0].legend()
axs[0].grid(alpha=0.3)

# Bottom panel: ratio
axs[1].plot(z, ratio, lw=2, color='black')
axs[1].set_xlabel('Redshift $z$')
axs[1].set_ylabel(r'$\chi(z) / D_A(z)$')
axs[1].grid(alpha=0.3)

# Save
fig.savefig("distance_comoving_DA_and_ratio.pdf", dpi=300, bbox_inches="tight")
fig.savefig("distance_comoving_DA_and_ratio.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# =============================================================================
# CELL 8b: kSZ²-21cm Cross-Correlation — Seed-Averaged Plots
# (No redshift binning anywhere — earlier "BINNED" label was a misnomer.)
# Three plots, all via save_pdf_png:
#   1. D_ℓ vs z at ℓ=3000, NO error bars       (mean across seeds)
#   2. D_ℓ vs z at ℓ=3000, WITH error bars     (σ_seeds ⊕ σ_meas)
#   3. D_ℓ vs ℓ at selected x_e                (seed-averaged)
# =============================================================================

from astropy.cosmology import FlatLambdaCDM

print("\n" + "="*70)
print("VISUALIZING kSZ²-21cm CROSS-CORRELATION (SEED-AVERAGED)")
print("="*70)

# Subdirectory for final plots
plot_dir_final = os.path.join(plot_dir, "plot_final_cell")
os.makedirs(plot_dir_final, exist_ok=True)
plot_dir_save = plot_dir_final

# =============================================================================
# Load cross_corr_results_all from cache if not already in memory
# =============================================================================
if ('cross_corr_results_all' not in dir()
        and 'cross_corr_results_all' not in globals()) \
   or len(cross_corr_results_all) == 0:

    print("cross_corr_results_all not in memory → loading from Cell 7 cache")
    cross_corr_results_all = {}
    for seed in RANDOM_SEEDS:
        cc_cache = os.path.join(
            main_cache_dir, f"seed_{seed}", f"cross_corr_seed{seed}.npy"
        )
        if os.path.exists(cc_cache):
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded seed {seed} "
                  f"({len(cross_corr_results_all[seed])} redshifts)")
        else:
            print(f"  ✗ No cache found for seed {seed}")
    print(f"  Loaded {len(cross_corr_results_all)}/{N_SEEDS} seeds")
else:
    print(f"cross_corr_results_all already in memory "
          f"({len(cross_corr_results_all)} seeds)")


if len(cross_corr_results_all) > 0:

    # =========================================================================
    # Convert k → ℓ for ALL seeds (keep per-realisation error)
    # =========================================================================
    print(f"\n=== CONVERTING TO ℓ-SPACE FOR ALL SEEDS ===")

    T_CMB_0_K = 2.725
    cosmo     = FlatLambdaCDM(H0=67.77, Om0=0.3086)

    cross_corr_ell_all = {}   # {seed: {z_node: ell_results_dict}}

    for seed, ccr in cross_corr_results_all.items():
        cross_corr_ell_results = {}

        for z_node in sorted(ccr.keys()):
            results          = ccr[z_node]
            D_A_Mpc          = float(cosmo.angular_diameter_distance(z_node).value)
            chi_comoving_Mpc = float(cosmo.comoving_distance(z_node).value)
            T_CMB_z_uK       = T_CMB_0_K * 1e6

            k_centers  = results['k_centers']
            ell_from_k = k_centers * chi_comoving_Mpc / 0.67

            C_cross_ell           = (results['C_cross_1d']
                                     * 0.67**2 / D_A_Mpc**2)
            C_cross_ell_err_total = (results['C_cross_1d_err_total']
                                     * 0.67**2 / D_A_Mpc**2)

            D_cross_ell           = (ell_from_k * (ell_from_k + 1)
                                     * C_cross_ell / (2 * np.pi))
            D_cross_ell_err_total = (ell_from_k * (ell_from_k + 1)
                                     * C_cross_ell_err_total / (2 * np.pi))

            D_cross_ell_uK_mK           = D_cross_ell           * T_CMB_z_uK**2
            D_cross_ell_uK_mK_err_total = D_cross_ell_err_total * T_CMB_z_uK**2

            P_kSZ2_ell = results['P_kSZ2_1d'] * 0.67**2 / D_A_Mpc**2
            P_T21_ell  = results['P_T21_1d']  * 0.67**2 / D_A_Mpc**2
            with np.errstate(divide='ignore', invalid='ignore'):
                r_cross = C_cross_ell / np.sqrt(P_kSZ2_ell * P_T21_ell)

            cross_corr_ell_results[z_node] = {
                'ell_from_k'                 : ell_from_k,
                'D_cross_ell_uK_mK'          : D_cross_ell_uK_mK,
                'D_cross_ell_uK_mK_err_total': D_cross_ell_uK_mK_err_total,
                'r_cross'                    : r_cross,
                'D_A_Mpc'                    : D_A_Mpc,
                'T_CMB_z_uK'                 : T_CMB_z_uK,
            }

        cross_corr_ell_all[seed] = cross_corr_ell_results

    print(f"Converted {len(cross_corr_ell_all)} seeds to ℓ-space")

    # Reference seed (for the node-redshift list and the x_e curve)
    ref_seed    = next(iter(cross_corr_ell_all))
    ref_lc      = lightcones[ref_seed]
    all_z_nodes = sorted(cross_corr_ell_all[ref_seed].keys())

    # =========================================================================
    # Seed-average D_ℓ(ℓ=3000) at every node redshift  (no redshift binning)
    # =========================================================================
    ell_target = 3000

    z_used        = []
    D_mean_per_z  = []
    sigma_seeds_per_z = []
    sigma_meas_per_z  = []
    sigma_total_per_z = []

    print(f"\n=== SEED-AVERAGING AT ℓ = {ell_target} ===")

    for z_target in all_z_nodes:
        D_seed_vals   = []
        D_err_sq_vals = []

        for seed, ell_res in cross_corr_ell_all.items():
            if z_target not in ell_res:
                continue
            res   = ell_res[z_target]
            ell   = res['ell_from_k']
            D_ell = res['D_cross_ell_uK_mK']
            D_err = res['D_cross_ell_uK_mK_err_total']
            idx   = np.argmin(np.abs(ell - ell_target))

            if np.isfinite(D_ell[idx]) and np.isfinite(D_err[idx]):
                D_seed_vals.append(D_ell[idx])
                D_err_sq_vals.append(D_err[idx]**2)

        if len(D_seed_vals) >= 2:
            D_mean      = np.mean(D_seed_vals)
            sigma_seeds = np.std(D_seed_vals, ddof=1)
            sigma_meas  = np.sqrt(np.mean(D_err_sq_vals))
            sigma_total = np.sqrt(sigma_seeds**2 + sigma_meas**2)

            z_used.append(z_target)
            D_mean_per_z.append(D_mean)
            sigma_seeds_per_z.append(sigma_seeds)
            sigma_meas_per_z.append(sigma_meas)
            sigma_total_per_z.append(sigma_total)

    z_used            = np.array(z_used)
    D_mean_per_z      = np.array(D_mean_per_z)
    sigma_seeds_per_z = np.array(sigma_seeds_per_z)
    sigma_meas_per_z  = np.array(sigma_meas_per_z)
    sigma_total_per_z = np.array(sigma_total_per_z)

    print(f"  Points: {len(z_used)} redshifts at ℓ ≈ {ell_target}")

    # =========================================================================
    # PLOT A: D_ℓ vs z at ℓ=3000 — NO error bars
    # =========================================================================
    print(f"\n=== PLOT A: D_ℓ vs z at ℓ={ell_target}, NO error bars ===")

    def _draw_A(ax):
        ax.plot(z_used, D_mean_per_z,
                color='darkred', lw=2.5, alpha=0.9,
                marker='o', markersize=6,
                label=f'$\\ell$={ell_target}')

        ax.set_xlabel(r'Redshift $z$')
        ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.legend(loc='best', framealpha=0.9)
        ax.invert_xaxis()

        ax.text(0.05, 0.95,
                f'{N_SEEDS} seeds | mean only',
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_A, plot_dir_save,
        f"kSZ2_21cm_cross_Dl_vs_z_ell{ell_target}_NOERR",
        title=(r'kSZ$^2$-21cm Cross-Power $D_\ell$ vs Redshift '
               f'at $\\ell$={ell_target} (mean, {N_SEEDS} seeds)'),
    )
    print(f"✓ Saved: kSZ2_21cm_cross_Dl_vs_z_ell{ell_target}_NOERR")

    # =========================================================================
    # PLOT B: D_ℓ vs z at ℓ=3000 — WITH error bars (σ_seeds ⊕ σ_meas)
    # =========================================================================
    print(f"\n=== PLOT B: D_ℓ vs z at ℓ={ell_target}, WITH error bars ===")

    def _draw_B(ax):
        ax.errorbar(z_used, D_mean_per_z, yerr=sigma_total_per_z,
                    color='darkred', lw=2.5, alpha=0.85,
                    marker='o', markersize=6,
                    capsize=4, capthick=1.5,
                    label=f'$\\ell$={ell_target}')
        ax.fill_between(z_used,
                        D_mean_per_z - sigma_total_per_z,
                        D_mean_per_z + sigma_total_per_z,
                        color='darkred', alpha=0.2)

        ax.set_xlabel(r'Redshift $z$')
        ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.legend(loc='best', framealpha=0.9)
        ax.invert_xaxis()

        ax.text(0.05, 0.95,
                f'{N_SEEDS} seeds | '
                r'$\sigma_{\rm seeds} \oplus \sigma_{\rm meas}$',
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_B, plot_dir_save,
        f"kSZ2_21cm_cross_Dl_vs_z_ell{ell_target}_ERR",
        title=(r'kSZ$^2$-21cm Cross-Power $D_\ell$ vs Redshift '
               f'at $\\ell$={ell_target} '
               f'({N_SEEDS} seeds, errors in quadrature)'),
    )
    print(f"✓ Saved: kSZ2_21cm_cross_Dl_vs_z_ell{ell_target}_ERR")

    # =========================================================================
    # PLOT C: D_ℓ vs ℓ at selected x_e (seed-averaged, errors in quadrature)
    # =========================================================================
    print(f"\n=== PLOT C: D_ℓ vs ℓ at selected x_e (seed-averaged) ===")

    z_nodes_sorted = ref_lc.node_redshifts[::-1]
    x_e_nodes      = 1.0 - ref_lc.global_xH[::-1]

    target_xe = [0.2, 0.5, 0.9]
    selected_z = [z_nodes_sorted[np.argmin(np.abs(x_e_nodes - xe))]
                  for xe in target_xe]
    selected_xe = [x_e_nodes[np.argmin(np.abs(x_e_nodes - xe))]
                   for xe in target_xe]

    print("  Selected redshifts:")
    for z, xe in zip(selected_z, selected_xe):
        print(f"    z={z:.2f}, x_e={xe:.3f}")

    colors_selected = ['blue', 'green', 'red']

    # Pre-compute per-(z_target) seed averages so the closure stays clean
    plotC_data = []   # list of (ell_ref[valid], D_mean[valid], sigma_total[valid], z_target, xe, color)

    for i, (z_target, xe) in enumerate(zip(selected_z, selected_xe)):

        D_ell_seeds   = []
        D_err_sq_list = []
        ell_ref       = None
        valid_ref     = None

        for seed, ell_res in cross_corr_ell_all.items():
            z_closest = min(ell_res.keys(), key=lambda z: abs(z - z_target))
            if abs(z_closest - z_target) > 0.5:
                continue
            res   = ell_res[z_closest]
            ell   = res['ell_from_k']
            D_ell = res['D_cross_ell_uK_mK']
            D_err = res['D_cross_ell_uK_mK_err_total']
            valid = ~np.isnan(D_ell) & np.isfinite(D_ell) & (ell > 10)
            if np.sum(valid) > 5:
                if ell_ref is None:
                    ell_ref   = ell
                    valid_ref = valid
                D_ell_seeds.append(D_ell)
                D_err_sq_list.append(D_err**2)

        if len(D_ell_seeds) == 0:
            continue

        D_matrix     = np.array(D_ell_seeds)
        D_err_sq_mat = np.array(D_err_sq_list)

        D_mean      = np.nanmean(D_matrix, axis=0)
        sigma_seeds = np.nanstd(D_matrix, ddof=1, axis=0)
        sigma_meas  = np.sqrt(np.nanmean(D_err_sq_mat, axis=0))
        sigma_total = np.sqrt(sigma_seeds**2 + sigma_meas**2)

        valid = valid_ref & ~np.isnan(D_mean)

        plotC_data.append((
            ell_ref[valid],
            D_mean[valid],
            sigma_total[valid],
            float(z_target),
            float(xe),
            colors_selected[i],
        ))

    def _draw_C(ax):
        for ell_v, D_v, sig_v, z_t, xe_t, color in plotC_data:
            ax.errorbar(ell_v, D_v, yerr=sig_v,
                        color=color, lw=2.5, alpha=0.8,
                        marker='o', markersize=5,
                        capsize=3, capthick=1.5,
                        label=f'z≈{z_t:.1f} ($x_e$={xe_t:.2f})',
                        errorevery=3)
            ax.fill_between(ell_v, D_v - sig_v, D_v + sig_v,
                            color=color, alpha=0.15)

        ax.set_xlabel(r'Multipole $\ell$')
        ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)')
        ax.set_xscale('log')
        ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
        ax.legend(loc='best', framealpha=0.9)

        ax.text(0.02, 0.02,
                f'{N_SEEDS} seeds | '
                r'$\sigma_{\rm seeds} \oplus \sigma_{\rm meas}$',
                transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    save_pdf_png(
        _draw_C, plot_dir_save,
        "kSZ2_21cm_cross_Dl_selected_xe_with_errors",
        title=(r'kSZ$^2$-21cm Cross-Power at Key Ionization Fractions'
               r' (Seed-Averaged)'),
    )
    print(f"✓ Saved: kSZ2_21cm_cross_Dl_selected_xe_with_errors")

    print("\n✓ ALL CELL 8b PLOTS COMPLETE")
    print(f"  Plots saved to: {plot_dir_save}")
    print(f"  1. kSZ2_21cm_cross_Dl_vs_z_ell{ell_target}_NOERR")
    print(f"  2. kSZ2_21cm_cross_Dl_vs_z_ell{ell_target}_ERR")
    print(f"  3. kSZ2_21cm_cross_Dl_selected_xe_with_errors")

else:
    print("\n✗ Skipping — no cross_corr_results_all available")

print("\n" + "="*70)


VISUALIZING kSZ²-21cm CROSS-CORRELATION (BINNED, ALL SEEDS)
Final plots directory exists: 18March2026_kSZ2_21cm/plots/plot_final_cell
cross_corr_results_all already in memory (60 seeds)

=== CONVERTING TO ℓ-SPACE FOR ALL SEEDS ===
Converted 60 seeds to ℓ-space

=== PLOT 1: D_ℓ Evolution (Seed-Averaged, errors in quadrature) ===
  Processing ℓ = 500...
    Points: 155
  Processing ℓ = 1000...
    Points: 155
  Processing ℓ = 3000...
    Points: 155
✓ Saved: kSZ2_21cm_cross_Dl_vs_z_fixed_ell_BINNED

=== PLOT 2: r vs z (Seed-Averaged, no node binning) ===
  Processing r(ℓ=500)...
    Points: 71
  Processing r(ℓ=1000)...
    Points: 71
  Processing r(ℓ=3000)...
    Points: 71
✓ Saved: kSZ2_21cm_cross_r_vs_z_fixed_ell_BINNED

=== PLOT 3: D_ℓ vs ℓ at selected x_e (Seed-Averaged) ===
  Selected redshifts:
    z=9.78, x_e=0.205
    z=8.02, x_e=0.508
    z=6.40, x_e=0.901


/var/tmp/pbs.1511489.swarm/ipykernel_124744/3976582252.py:304: RuntimeWarning: Mean of empty slice
  D_mean        = np.nanmean(D_matrix, axis=0)
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/tmp/pbs.1511489.swarm/ipykernel_124744/3976582252.py:306: RuntimeWarning: Mean of empty slice
  sigma_meas    = np.sqrt(np.nanmean(D_err_sq_mat, axis=0))


✓ Saved: kSZ2_21cm_cross_Dl_selected_xe_with_errors

✓ ALL VISUALIZATION COMPLETE!
  Plots saved to: 18March2026_kSZ2_21cm/plots/plot_final_cell
  1. kSZ2_21cm_cross_Dl_vs_z_fixed_ell_BINNED.png  ← D_ℓ vs z
  2. kSZ2_21cm_cross_r_vs_z_fixed_ell_BINNED.png   ← r vs z
  3. kSZ2_21cm_cross_Dl_selected_xe_with_errors.png ← D_ℓ vs ℓ



In [11]:
# =============================================================================
# CELL 8c: Redshift Evolution of 21cm Auto Power Spectrum (All Seeds)
# Consistent with Cell 8b: mean ± std across seeds (cosmic variance)
# =============================================================================

print("\n" + "="*70)
print("VISUALIZING 21cm AUTO POWER SPECTRUM REDSHIFT EVOLUTION")
print("="*70)

# Create subdirectory if needed
plot_dir_final = f"{plot_dir}/plot_final_cell"
if not os.path.exists(plot_dir_final):
    os.makedirs(plot_dir_final)
plot_dir_save = plot_dir_final

# ==========================================================================
# Load cross_corr_results_all from cache if not in memory
# ==========================================================================
if 'cross_corr_results_all' not in dir() or len(cross_corr_results_all) == 0:
    print("cross_corr_results_all not in memory → loading from Cell 7 cache")
    cross_corr_results_all = {}
    for seed in RANDOM_SEEDS:
        cc_cache = f"{cache_dir}/seed_{seed}/cross_corr_seed{seed}.npy"
        if os.path.exists(cc_cache):
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded seed {seed} "
                  f"({len(cross_corr_results_all[seed])} redshifts)")
        else:
            print(f"  ✗ No cache found for seed {seed}")
    print(f"  Loaded {len(cross_corr_results_all)}/{N_SEEDS} seeds")
else:
    print(f"cross_corr_results_all already in memory "
          f"({len(cross_corr_results_all)} seeds)")

if len(cross_corr_results_all) > 0:

    from astropy.cosmology import FlatLambdaCDM
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)

    # ==========================================================================
    # Convert P_T21(k) → D_ℓ^{21} for all seeds
    # ==========================================================================

    print(f"\n=== CONVERTING 21cm AUTO POWER TO ℓ-SPACE (ALL SEEDS) ===")

    auto_T21_ell_all = {}

    for seed, ccr in cross_corr_results_all.items():
        auto_T21_ell_results = {}

        for z_obs in sorted(ccr.keys()):
            results      = ccr[z_obs]
            D_A_Mpc      = float(cosmo.angular_diameter_distance(z_obs).value)
            chi_comoving = float(cosmo.comoving_distance(z_obs).value)

            k_centers  = results['k_centers']
            P_T21      = results['P_T21_1d']
            n_modes    = results['n_modes']

            ell_from_k = k_centers * chi_comoving / 0.67
            C_T21_ell  = P_T21 * 0.67**2 / D_A_Mpc**2
            D_T21_ell  = ell_from_k * (ell_from_k + 1) * C_T21_ell / (2 * np.pi)

            with np.errstate(divide='ignore', invalid='ignore'):
                err_frac = np.where(n_modes > 0, 1.0 / np.sqrt(n_modes), np.nan)

            D_T21_ell_err = np.abs(D_T21_ell) * err_frac

            auto_T21_ell_results[z_obs] = {
                'ell_from_k'   : ell_from_k,
                'D_T21_ell'    : D_T21_ell,
                'D_T21_ell_err': D_T21_ell_err,
            }

        auto_T21_ell_all[seed] = auto_T21_ell_results

    print(f"Converted {len(auto_T21_ell_all)} seeds to ℓ-space")

    # ==========================================================================
    # Average across seeds at each redshift (mean ± std)
    # ==========================================================================

    ref_seed = list(auto_T21_ell_all.keys())[0]
    all_z    = sorted(auto_T21_ell_all[ref_seed].keys())

    auto_T21_ell_averaged = {}

    for z_obs in all_z:
        D_seeds = []
        ell_ref = None

        for seed, res_dict in auto_T21_ell_all.items():
            if z_obs not in res_dict:
                continue

            res   = res_dict[z_obs]
            ell   = res['ell_from_k']
            D_ell = res['D_T21_ell']

            if ell_ref is None:
                ell_ref = ell

            D_seeds.append(D_ell)

        if len(D_seeds) == 0:
            continue

        D_matrix = np.array(D_seeds)

        D_mean = np.nanmean(D_matrix, axis=0)
        D_std  = np.nanstd(D_matrix, ddof=1, axis=0)

        auto_T21_ell_averaged[z_obs] = {
            'ell_from_k'   : ell_ref,
            'D_T21_ell'    : D_mean,
            'D_T21_ell_err': D_std,
        }

    print(f"Averaged over {len(auto_T21_ell_all)} seeds "
          f"at {len(auto_T21_ell_averaged)} redshifts")

    # ==========================================================================
    # Plot: D_ℓ^{21} vs ℓ, colored by redshift
    # ==========================================================================

    print(f"\n=== GENERATING 21cm AUTO POWER SPECTRUM PLOT ===")

    z_all = np.array(sorted(auto_T21_ell_averaged.keys()))
    norm  = mcolors.Normalize(vmin=z_all.min(), vmax=z_all.max())
    cmap  = cm.plasma

    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for z_obs in z_all:
        res   = auto_T21_ell_averaged[z_obs]
        ell   = res['ell_from_k']
        D_ell = res['D_T21_ell']
        D_err = res['D_T21_ell_err']

        valid = (~np.isnan(D_ell) & np.isfinite(D_ell)
                 & (ell > 10) & (D_ell > 0))

        if np.sum(valid) < 3:
            continue

        color = cmap(norm(z_obs))

        ax.plot(ell[valid], D_ell[valid],
                color=color, lw=1.5, alpha=0.75)

        ax.fill_between(ell[valid],
                        D_ell[valid] - D_err[valid],
                        D_ell[valid] + D_err[valid],
                        color=color, alpha=0.12)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'Redshift $z$')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'Multipole $\ell$')
    ax.set_ylabel(r'$D_\ell^{21}\ [\mathrm{mK}^2]$')

    ax.text(0.05, 0.05,
            f'Shaded: seed-to-seed scatter\n'
            f'{N_SEEDS} seeds averaged\n'
            r'$D_\ell = \ell(\ell+1)C_\ell/2\pi$',
            transform=ax.transAxes, fontsize=11,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "21cm_auto_Dl_vs_ell_redshift_evolution"

    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')

    ax.set_title(r'21cm Auto Power Spectrum $D_\ell^{21}$ — Redshift Evolution'
                 f' ({N_SEEDS} seeds)',
                 fontweight='bold')

    fig.savefig(f"{plot_dir_save}/{plot_name}.png",
                dpi=300, bbox_inches='tight')

    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    print("\n✓ 21cm AUTO POWER SPECTRUM PLOT COMPLETE")
    print(f"  Saved to: {plot_dir_save}")

else:
    print("\n✗ Skipping — no cross_corr_results_all available")

print("\n" + "="*70)


VISUALIZING 21cm AUTO POWER SPECTRUM REDSHIFT EVOLUTION
cross_corr_results_all already in memory (60 seeds)

=== CONVERTING 21cm AUTO POWER TO ℓ-SPACE (ALL SEEDS) ===
Converted 60 seeds to ℓ-space
Averaged over 60 seeds at 155 redshifts

=== GENERATING 21cm AUTO POWER SPECTRUM PLOT ===


/var/tmp/pbs.1511489.swarm/ipykernel_124744/3801180997.py:114: RuntimeWarning: Mean of empty slice
  D_mean = np.nanmean(D_matrix, axis=0)
/user1/swanith/.conda/envs/p21c_v3/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


✓ Saved: 21cm_auto_Dl_vs_ell_redshift_evolution

✓ 21cm AUTO POWER SPECTRUM PLOT COMPLETE
  Saved to: 18March2026_kSZ2_21cm/plots/plot_final_cell

